# PortPy `debug.ipynb` -- Full Package Function Reference

A point-by-point exercise of **every public, callable function and class in `src/portpy/`**,
run against **real market data** spanning several distinct trading calendars: US equities,
a REIT (real estate), a Treasury-bond ETF, a commodity ETF, and 24/7 crypto -- plus a
benchmark and a live risk-free rate.

Every section below is numbered `N.M` and maps to one function (or one closely related
group of calls on the same function), so you can search for e.g. `10.5 value_at_risk` and
see every parameter combination that function supports, not just a single default call.

**Coverage:**

| Package area | Sections |
|---|---|
| Data acquisition & cleaning | 1.x |
| `portpy.core` (calendars, currency, weights, asset classes) | 2.x - 5.x |
| `portpy.utils.validation` (incl. deliberate failure cases) | 6.x |
| `portpy.Portfolio` (the main entry point) | 7.x |
| `portpy.metrics.returns` | 8.x |
| `portpy.metrics.risk` | 9.x |
| `portpy.metrics.performance` | 10.x |
| `portpy.metrics.drawdowns` | 11.x |
| `portpy.metrics.rolling` | 12.x |
| `portpy.metrics.distributions` | 13.x |
| `portpy.metrics.benchmarks` | 14.x |
| `portpy.metrics.regressions` | 15.x |
| `portpy.metrics.covariance` | 16.x |
| `portpy.metrics.summary` | 17.x |
| `portpy.metrics.costs` | 18.x |
| `portpy.explain` (the explainability layer) | 19.x |
| `Portfolio.metrics` auto-fill mechanics | 20.x |
| Coverage audit + not-yet-implemented subpackages | 21.x |

Data sources: **yfinance** (no API key required, so this notebook runs for anyone who
clones the repo).

## 1. Data Acquisition & Cleaning

### 1.1 Universe definition

Deliberately spans every trading-duration case PortPy calls out in its docs:

- **Equities** (Mon-Fri): `AAPL`, `MSFT`, `JPM`
- **Real estate** (Mon-Fri, via REIT ETF): `VNQ`
- **Bond** (Mon-Fri, via long-Treasury ETF): `TLT`
- **Commodity** (Mon-Fri, via gold ETF): `GLD`
- **Crypto** (7 days/week): `BTC-USD`, `ETH-USD`
- **International equity, EUR-denominated**: `SAP.DE` (+ `EURUSD=X` for currency conversion)
- **Benchmark**: `SPY`
- **Risk-free rate**: `^IRX` (13-week T-Bill yield)

In [1]:
from __future__ import annotations

import warnings
from collections.abc import Callable

import numpy as np
import pandas as pd
import yfinance as yf

warnings.filterwarnings("ignore")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

import portpy
from portpy import Portfolio
from portpy import explain as pexplain
from portpy.core import (
    ALWAYS_ON_CLASSES,
    AssetClass,
    align_calendars,
    calendar_coverage_report,
    convert_to_base_currency,
    detect_frequency,
    equal_weights,
    normalize_weights,
)
from portpy.explain import Explanation, MetricResult, available, get, register
from portpy.utils import validation as pv
from portpy.utils.constants import (
    CALENDAR_DAYS_PER_YEAR,
    DEFAULT_CONFIDENCE_LEVEL,
    DEFAULT_MAR,
    DEFAULT_RISK_FREE_RATE,
    EPSILON,
    MONTHS_PER_YEAR,
    QUARTERS_PER_YEAR,
    TRADING_DAYS_PER_YEAR,
    WEEKS_PER_YEAR,
)

print("PortPy version:", portpy.__version__)
print(
    "Constants -> TRADING_DAYS_PER_YEAR:", TRADING_DAYS_PER_YEAR,
    "| CALENDAR_DAYS_PER_YEAR:", CALENDAR_DAYS_PER_YEAR,
    "| WEEKS_PER_YEAR:", WEEKS_PER_YEAR,
    "| MONTHS_PER_YEAR:", MONTHS_PER_YEAR,
    "| QUARTERS_PER_YEAR:", QUARTERS_PER_YEAR,
)
print(
    "DEFAULT_RISK_FREE_RATE:", DEFAULT_RISK_FREE_RATE,
    "| DEFAULT_CONFIDENCE_LEVEL:", DEFAULT_CONFIDENCE_LEVEL,
    "| DEFAULT_MAR:", DEFAULT_MAR,
    "| EPSILON:", EPSILON,
)

# Tracks every portpy.metrics.__all__ name we actually exercise below, checked in section 21.1.
COVERED: set[str] = set()

# Tracks every name we actually rendered through the explain() layer (.explain() on a live
# MetricResult, or pexplain("name") directly) -- checked in section 21.1. This is the whole
# point of this notebook: not just "the function ran once", but "its explanation card renders".
EXPLAINED: set[str] = set()


def note(*names: str) -> None:
    COVERED.update(names)


def note_explained(*names: str) -> None:
    EXPLAINED.update(names)

PortPy version: 0.1.0
Constants -> TRADING_DAYS_PER_YEAR: 252 | CALENDAR_DAYS_PER_YEAR: 365 | WEEKS_PER_YEAR: 52 | MONTHS_PER_YEAR: 12 | QUARTERS_PER_YEAR: 4
DEFAULT_RISK_FREE_RATE: 0.0 | DEFAULT_CONFIDENCE_LEVEL: 0.95 | DEFAULT_MAR: 0.0 | EPSILON: 1e-12


### 1.2 Fetch equities / REIT / bond / commodity prices (Mon-Fri calendar)

In [2]:
PERIOD = "3y"

EQUITY_LIKE = ["AAPL", "MSFT", "JPM", "VNQ", "TLT", "GLD"]
raw_equity_like = yf.download(EQUITY_LIKE, period=PERIOD, auto_adjust=True, progress=False)["Close"]
print(f"Fetched {raw_equity_like.shape[1]} symbols, {raw_equity_like.shape[0]} rows "
      f"({raw_equity_like.index.min().date()} -> {raw_equity_like.index.max().date()})")
raw_equity_like.tail()

Fetched 6 symbols, 752 rows (2023-08-07 -> 2026-08-05)


Ticker,AAPL,GLD,JPM,MSFT,TLT,VNQ
Date,,,,,,
2026-07-30,333.4300,377.1600,350.8500,451.1000,82.4678,99.4900
2026-07-31,308.9100,371.5400,351.7900,464.7200,81.9200,98.9500
2026-08-03,303.4200,371.7100,352.6400,487.6500,82.1900,99.0700
2026-08-04,309.3800,374.1600,357.5200,492.8100,82.8200,98.9200
2026-08-05,311.0000,389.6400,359.2400,487.4600,83.0000,98.9200


### 1.3 Fetch crypto prices (7 days/week calendar)

In [3]:
CRYPTO = ["BTC-USD", "ETH-USD"]
raw_crypto = yf.download(CRYPTO, period=PERIOD, auto_adjust=True, progress=False)["Close"]
raw_crypto = raw_crypto.rename(columns=lambda c: c.replace("-USD", ""))
print(f"Fetched {raw_crypto.shape[1]} crypto symbols, {raw_crypto.shape[0]} rows "
      f"({raw_crypto.index.min().date()} -> {raw_crypto.index.max().date()})")
raw_crypto.tail()

Fetched 2 crypto symbols, 1096 rows (2023-08-06 -> 2026-08-06)


Ticker,BTC,ETH
Date,,
2026-08-01,"62,763.3203","1,843.4177"
2026-08-02,"63,482.0000","1,882.5220"
2026-08-03,"63,460.8984","1,858.2565"
2026-08-04,"64,055.9531","1,868.3859"
2026-08-06,"64,475.9883","1,894.9600"


### 1.4 Fetch international equity (EUR) + FX rate, and the benchmark

In [4]:
sap_eur_raw = yf.download("SAP.DE", period=PERIOD, auto_adjust=True, progress=False)["Close"].iloc[:, 0].rename("SAP")
eurusd_raw = yf.download("EURUSD=X", period=PERIOD, auto_adjust=True, progress=False)["Close"].iloc[:, 0].rename("EUR")
benchmark_prices_raw = yf.download("SPY", period=PERIOD, auto_adjust=True, progress=False)["Close"].iloc[:, 0].rename("SPY")

# Data cleaning: a same-day fetch can catch a not-yet-settled last bar (NaN) on a foreign
# exchange with a different close time than the US market -- drop it rather than silently
# propagating a NaN into every downstream calculation.
sap_eur = sap_eur_raw.dropna()
eurusd = eurusd_raw.dropna()
benchmark_prices = benchmark_prices_raw.dropna()
print(f"SAP.DE: {len(sap_eur)} rows ({sap_eur_raw.isna().sum()} NaN row(s) dropped), last EUR close={sap_eur.iloc[-1]:.2f}")
print(f"EURUSD=X: {len(eurusd)} rows ({eurusd_raw.isna().sum()} NaN row(s) dropped), last rate={eurusd.iloc[-1]:.4f}")
print(f"SPY benchmark: {len(benchmark_prices)} rows ({benchmark_prices_raw.isna().sum()} NaN row(s) dropped), "
      f"last close={benchmark_prices.iloc[-1]:.2f}")

SAP.DE: 760 rows (1 NaN row(s) dropped), last EUR close=167.38
EURUSD=X: 779 rows (0 NaN row(s) dropped), last rate=1.1557
SPY benchmark: 752 rows (0 NaN row(s) dropped), last close=769.79


### 1.5 Fetch the risk-free rate (`^IRX`, 13-week T-Bill yield)

In [5]:
irx = yf.download("^IRX", period="1mo", auto_adjust=True, progress=False)["Close"].iloc[:, 0]
risk_free_rate = float(irx.iloc[-1]) / 100.0
print(f"Latest 13-week T-Bill yield (^IRX): {risk_free_rate:.4%} annual -- used as the portfolio's risk_free_rate")

Latest 13-week T-Bill yield (^IRX): 3.7250% annual -- used as the portfolio's risk_free_rate


### 1.6 Inspect raw data quality before doing anything about it

Never trust that two data feeds line up. `AAPL`/`MSFT`/`JPM`/`VNQ`/`TLT`/`GLD` all trade on
the same NYSE/Nasdaq calendar so there should be no cross-asset gaps -- check anyway.

In [6]:
n_missing = raw_equity_like.isna().sum()
print("Missing values per symbol (equity-like group):")
print(n_missing)
assert not n_missing.any(), "Unexpected NaNs within the equity-like group -- investigate before proceeding."
print("\nNo missing values within the equity-like group. Good.")

Missing values per symbol (equity-like group):
Ticker
AAPL    0
GLD     0
JPM     0
MSFT    0
TLT     0
VNQ     0
dtype: int64

No missing values within the equity-like group. Good.


## 2. Currency Conversion (`portpy.core.currency`)

### 2.1 `convert_to_base_currency` -- bring SAP.DE (EUR) into USD

In [7]:
sap_prices = sap_eur.to_frame()
fx_frame = eurusd.to_frame()

sap_usd_frame = convert_to_base_currency(
    sap_prices, fx_frame, asset_currencies={"SAP": "EUR"}, base_currency="USD"
)
sap_usd = sap_usd_frame["SAP"].rename("SAP")

print(f"SAP.DE EUR close (last): {sap_eur.iloc[-1]:.2f}  |  EURUSD (last): {eurusd.iloc[-1]:.4f}")
print(f"SAP.DE converted to USD (last): {sap_usd.iloc[-1]:.2f}")
assert np.isclose(sap_usd.iloc[-1], sap_eur.iloc[-1] * eurusd.reindex(sap_eur.index).ffill().iloc[-1])

# Assets already in the base currency (or missing from asset_currencies) are left untouched.
passthrough = convert_to_base_currency(
    raw_equity_like[["AAPL"]], fx_frame, asset_currencies={"AAPL": "USD"}, base_currency="USD"
)
assert passthrough["AAPL"].equals(raw_equity_like["AAPL"])
print("Assets already in the base currency are left untouched (AAPL check passed).")

# Missing FX column for a referenced currency raises, rather than silently skipping.
try:
    convert_to_base_currency(sap_prices, fx_frame, asset_currencies={"SAP": "GBP"}, base_currency="USD")
except ValueError as exc:
    print(f"\nExpected ValueError (no GBP column in fx_rates): {exc}")

SAP.DE EUR close (last): 167.38  |  EURUSD (last): 1.1557
SAP.DE converted to USD (last): 192.61
Assets already in the base currency are left untouched (AAPL check passed).

Expected ValueError (no GBP column in fx_rates): No FX rate column found for currency 'GBP' (needed by asset 'SAP'). fx_rates columns: ['EUR']


## 3. Calendar Alignment (`portpy.core.calendar`)

PortPy never aligns data for you (see the package README) -- these are the tools you call
*before* constructing a `Portfolio`.

### 3.1 `detect_frequency` -- classify each group's spacing

In [8]:
print("detect_frequency(equity-like):", detect_frequency(raw_equity_like.index))
print("detect_frequency(crypto):     ", detect_frequency(raw_crypto.index))
print("detect_frequency(SAP.DE):     ", detect_frequency(sap_usd.index))

# Edge cases: too few points, and a monthly-spaced index.
print("detect_frequency(<3 points):  ", detect_frequency(raw_equity_like.index[:2]))
monthly_idx = pd.date_range("2020-01-31", periods=6, freq="ME")
print("detect_frequency(monthly):    ", detect_frequency(monthly_idx))

detect_frequency(equity-like): daily-5
detect_frequency(crypto):      daily-7
detect_frequency(SAP.DE):      daily-5
detect_frequency(<3 points):   unknown
detect_frequency(monthly):     monthly


### 3.2 `calendar_coverage_report` -- see the mismatch before resolving it

In [9]:
combined_raw = pd.concat([raw_equity_like, raw_crypto, sap_usd], axis=1, sort=True)
coverage = calendar_coverage_report(combined_raw)
print(coverage)
print(f"\nRows with >=1 NaN: {int(combined_raw.isna().any(axis=1).sum())} / {len(combined_raw)}")

      frequency first_date  last_date  n_observations  n_missing_in_frame
asset                                                                    
AAPL    daily-5 2023-08-07 2026-08-05             752                 345
GLD     daily-5 2023-08-07 2026-08-05             752                 345
JPM     daily-5 2023-08-07 2026-08-05             752                 345
MSFT    daily-5 2023-08-07 2026-08-05             752                 345
TLT     daily-5 2023-08-07 2026-08-05             752                 345
VNQ     daily-5 2023-08-07 2026-08-05             752                 345
BTC     daily-7 2023-08-06 2026-08-06            1096                   1
ETH     daily-7 2023-08-06 2026-08-06            1096                   1
SAP     daily-5 2023-08-07 2026-08-04             760                 337

Rows with >=1 NaN: 359 / 1097


### 3.3 `align_calendars` -- all three resolution methods, compared

In [10]:
aligned_intersection = align_calendars(combined_raw, method="intersection")
aligned_ffill_union = align_calendars(combined_raw, method="ffill_union")
aligned_business_days = align_calendars(combined_raw, method="business_days")

for label, df in [
    ("intersection", aligned_intersection),
    ("ffill_union", aligned_ffill_union),
    ("business_days", aligned_business_days),
]:
    assert df.isna().sum().sum() == 0
    print(f"'{label}': {len(df)} rows, {df.index.min().date()} -> {df.index.max().date()}, no NaNs remaining")

# Also accepts a dict of {name: Series} instead of a pre-combined DataFrame.
from_dict = align_calendars({"AAPL": raw_equity_like["AAPL"], "BTC": raw_crypto["BTC"]}, method="intersection")
print(f"\nalign_calendars() from a dict input: {from_dict.shape}")

# An unknown method raises immediately rather than silently falling back.
try:
    align_calendars(combined_raw, method="nearest")
except ValueError as exc:
    print(f"Expected ValueError (bad method): {exc}")

'intersection': 738 rows, 2023-08-07 -> 2026-08-04, no NaNs remaining
'ffill_union': 1096 rows, 2023-08-07 -> 2026-08-06, no NaNs remaining
'business_days': 784 rows, 2023-08-07 -> 2026-08-06, no NaNs remaining

align_calendars() from a dict input: (751, 2)
Expected ValueError (bad method): method must be one of ('intersection', 'ffill_union', 'business_days'), got 'nearest'.


In [11]:
# We keep crypto's 24/7 calendar (ffill_union) so the final portfolio genuinely "never sleeps" --
# equity/REIT/bond/commodity/SAP prices are simply carried forward through the weekend.
master = aligned_ffill_union.copy()
print(f"Master frame for the Portfolio: {master.shape[1]} assets x {master.shape[0]} days "
      f"({master.index.min().date()} -> {master.index.max().date()})")
master.tail()

Master frame for the Portfolio: 9 assets x 1096 days (2023-08-07 -> 2026-08-06)


,AAPL,GLD,JPM,MSFT,TLT,VNQ,BTC,ETH,SAP
Date,,,,,,,,,
2026-08-02,308.9100,371.5400,351.7900,464.7200,81.9200,98.9500,"63,482.0000","1,882.5220",181.7029
2026-08-03,303.4200,371.7100,352.6400,487.6500,82.1900,99.0700,"63,460.8984","1,858.2565",190.0606
2026-08-04,309.3800,374.1600,357.5200,492.8100,82.8200,98.9200,"64,055.9531","1,868.3859",192.6100
2026-08-05,311.0000,389.6400,359.2400,487.4600,83.0000,98.9200,"64,055.9531","1,868.3859",192.6100
2026-08-06,311.0000,389.6400,359.2400,487.4600,83.0000,98.9200,"64,475.9883","1,894.9600",192.6100


## 4. Portfolio Weights (`portpy.core.weights`)

### 4.1 `equal_weights`

In [12]:
eq_w = equal_weights(list(master.columns))
print(eq_w)
assert np.isclose(eq_w.sum(), 1.0)

try:
    equal_weights([])
except ValueError as exc:
    print(f"\nExpected ValueError (zero assets): {exc}")

AAPL   0.1111
GLD    0.1111
JPM    0.1111
MSFT   0.1111
TLT    0.1111
VNQ    0.1111
BTC    0.1111
ETH    0.1111
SAP    0.1111
Name: weight, dtype: float64

Expected ValueError (zero assets): Cannot build weights for zero assets.


### 4.2 `normalize_weights` -- dict / list / array inputs, plus long/short and error cases

In [13]:
target_weights = {
    "AAPL": 0.15, "MSFT": 0.13, "JPM": 0.10, "SAP": 0.07,
    "VNQ": 0.10, "TLT": 0.10, "GLD": 0.10, "BTC": 0.15, "ETH": 0.10,
}
assert np.isclose(sum(target_weights.values()), 1.0)

# From a dict.
w_from_dict = normalize_weights(target_weights, names=list(master.columns))
print("From dict:\n", w_from_dict)

# From a plain list/array, using `names=` for ordering (unnormalized input -- rescaled to sum to 1).
w_from_list = normalize_weights([1, 1, 1, 1], names=["A", "B", "C", "D"])
print("\nFrom unnormalized list [1,1,1,1]:\n", w_from_list)

# From a numpy array.
w_from_array = normalize_weights(np.array([2.0, 1.0, 1.0]), names=["X", "Y", "Z"])
print("\nFrom numpy array [2,1,1]:\n", w_from_array)

# Long/short: negative weights are allowed by default and simply rescale around the net exposure.
long_short = {"A": 0.6, "B": 0.6, "C": -0.2}
w_long_short = normalize_weights(long_short)
print("\nLong/short (allow_negative=True, default):\n", w_long_short)

# allow_negative=False raises on a book with shorts (e.g. for a long-only optimizer).
try:
    normalize_weights(long_short, allow_negative=False)
except ValueError as exc:
    print(f"\nExpected ValueError (negative weight, allow_negative=False): {exc}")

# Missing an asset that `names` requires raises.
try:
    normalize_weights({"A": 0.5, "B": 0.5}, names=["A", "B", "C"])
except ValueError as exc:
    print(f"Expected ValueError (missing asset C): {exc}")

# Weights summing to zero cannot be rescaled.
try:
    normalize_weights({"A": 1.0, "B": -1.0})
except ValueError as exc:
    print(f"Expected ValueError (weights sum to zero): {exc}")

From dict:
 AAPL   0.1500
GLD    0.1000
JPM    0.1000
MSFT   0.1300
TLT    0.1000
VNQ    0.1000
BTC    0.1500
ETH    0.1000
SAP    0.0700
Name: weight, dtype: float64

From unnormalized list [1,1,1,1]:
 A   0.2500
B   0.2500
C   0.2500
D   0.2500
Name: weight, dtype: float64

From numpy array [2,1,1]:
 X   0.5000
Y   0.2500
Z   0.2500
Name: weight, dtype: float64

Long/short (allow_negative=True, default):
 A    0.6000
B    0.6000
C   -0.2000
Name: weight, dtype: float64

Expected ValueError (negative weight, allow_negative=False): Negative weights are not allowed here (long-only constraint).
Expected ValueError (missing asset C): Weights are missing for assets: ['C']
Expected ValueError (weights sum to zero): Weights sum to zero; cannot normalize.


## 5. Asset Tagging (`portpy.core.asset`)

### 5.1 `AssetClass` -- full enum tour

In [14]:
for member in AssetClass:
    print(f"  {member.name:12s} = {member.value!r:14s}  always-on (24/7)? {member in ALWAYS_ON_CLASSES}")

asset_classes = {
    "AAPL": AssetClass.EQUITY, "MSFT": AssetClass.EQUITY, "JPM": AssetClass.EQUITY, "SAP": AssetClass.EQUITY,
    "VNQ": AssetClass.REAL_ESTATE,
    "TLT": AssetClass.BOND,
    "GLD": AssetClass.COMMODITY,
    "BTC": AssetClass.CRYPTO, "ETH": AssetClass.CRYPTO,
}
print("\nasset_classes for our master portfolio:")
for asset, cls in asset_classes.items():
    print(f"  {asset:6s} -> {cls.value}")

  EQUITY       = 'equity'        always-on (24/7)? False
  CRYPTO       = 'crypto'        always-on (24/7)? True
  BOND         = 'bond'          always-on (24/7)? False
  COMMODITY    = 'commodity'     always-on (24/7)? False
  FX           = 'fx'            always-on (24/7)? True
  REAL_ESTATE  = 'real_estate'   always-on (24/7)? False
  OTHER        = 'other'         always-on (24/7)? False

asset_classes for our master portfolio:
  AAPL   -> equity
  MSFT   -> equity
  JPM    -> equity
  SAP    -> equity
  VNQ    -> real_estate
  TLT    -> bond
  GLD    -> commodity
  BTC    -> crypto
  ETH    -> crypto


### 5.2 `ALWAYS_ON_CLASSES`

In [15]:
print("ALWAYS_ON_CLASSES:", {c.value for c in ALWAYS_ON_CLASSES})
print("CRYPTO always-on:", AssetClass.CRYPTO in ALWAYS_ON_CLASSES)
print("FX always-on:    ", AssetClass.FX in ALWAYS_ON_CLASSES)
print("EQUITY always-on:", AssetClass.EQUITY in ALWAYS_ON_CLASSES)

ALWAYS_ON_CLASSES: {'fx', 'crypto'}
CRYPTO always-on: True
FX always-on:     True
EQUITY always-on: False


## 6. Input Validation (`portpy.utils.validation`)

These helpers back every metric function and `Portfolio` itself. Each one below is
exercised on a **valid** call *and* on the failure case(s) it's meant to catch.

### 6.1 `ensure_datetime_index`

In [16]:
# Valid: returns the data unchanged.
result = pv.ensure_datetime_index(master, name="master")
assert result is master
print("Valid DatetimeIndex: passes through unchanged.")

# TypeError: not a DatetimeIndex at all.
bad_index_df = pd.DataFrame({"x": [1, 2, 3]})
try:
    pv.ensure_datetime_index(bad_index_df, name="bad_index_df")
except TypeError as exc:
    print(f"\nExpected TypeError (RangeIndex, not DatetimeIndex): {exc}")

# ValueError: duplicate dates.
dupes = pd.DataFrame(
    {"x": [1, 2, 3]}, index=pd.DatetimeIndex(["2024-01-01", "2024-01-01", "2024-01-02"])
)
try:
    pv.ensure_datetime_index(dupes, name="dupes")
except ValueError as exc:
    print(f"Expected ValueError (duplicate dates): {exc}")

# ValueError: unsorted index.
unsorted = pd.DataFrame(
    {"x": [1, 2, 3]}, index=pd.DatetimeIndex(["2024-01-03", "2024-01-01", "2024-01-02"])
)
try:
    pv.ensure_datetime_index(unsorted, name="unsorted")
except ValueError as exc:
    print(f"Expected ValueError (unsorted index): {exc}")

Valid DatetimeIndex: passes through unchanged.

Expected TypeError (RangeIndex, not DatetimeIndex): bad_index_df must be indexed by a pandas DatetimeIndex, got RangeIndex. Use `df.index = pd.to_datetime(df.index)` before passing data to PortPy.
Expected ValueError (duplicate dates): dupes has duplicate dates: [Timestamp('2024-01-01 00:00:00')]. PortPy does not silently drop duplicates - deduplicate explicitly first.
Expected ValueError (unsorted index): unsorted index must be sorted ascending. Call `.sort_index()` first.


### 6.2 `ensure_min_observations`

In [17]:
pv.ensure_min_observations(master, min_obs=2, name="master")
print("Valid: 2+ observations required, master has plenty -- no error, returns None.")

try:
    pv.ensure_min_observations(master.iloc[:1], min_obs=2, name="single_row")
except ValueError as exc:
    print(f"Expected ValueError (only 1 observation, need 2): {exc}")

Valid: 2+ observations required, master has plenty -- no error, returns None.
Expected ValueError (only 1 observation, need 2): single_row needs at least 2 observation(s), got 1.


### 6.3 `validate_confidence`

In [18]:
for c in (0.90, 0.95, 0.99):
    pv.validate_confidence(c)
    print(f"validate_confidence({c}) -> OK")

for bad in (0.0, 1.0, 1.5, -0.2):
    try:
        pv.validate_confidence(bad)
    except ValueError as exc:
        print(f"validate_confidence({bad}) -> Expected ValueError: {exc}")

validate_confidence(0.9) -> OK
validate_confidence(0.95) -> OK
validate_confidence(0.99) -> OK
validate_confidence(0.0) -> Expected ValueError: confidence must be in (0, 1), got 0.0.
validate_confidence(1.0) -> Expected ValueError: confidence must be in (0, 1), got 1.0.
validate_confidence(1.5) -> Expected ValueError: confidence must be in (0, 1), got 1.5.
validate_confidence(-0.2) -> Expected ValueError: confidence must be in (0, 1), got -0.2.


### 6.4 `to_series`

In [19]:
aapl_series = master["AAPL"]

# Series passthrough.
s1 = pv.to_series(aapl_series)
assert s1 is aapl_series
print("Series input -> passthrough, unchanged.")

# Single-column DataFrame -> squeezed to a Series.
s2 = pv.to_series(master[["AAPL"]])
print("Single-column DataFrame -> Series:", type(s2).__name__, s2.name)

# Multi-column DataFrame + explicit column -> that column as a Series.
s3 = pv.to_series(master, column="MSFT")
assert s3.equals(master["MSFT"])
print("Multi-column DataFrame + column='MSFT' -> matches master['MSFT']:", s3.equals(master["MSFT"]))

# Multi-column DataFrame, no column specified -> raises.
try:
    pv.to_series(master, name="master")
except ValueError as exc:
    print(f"Expected ValueError (multi-column, no column specified): {exc}")

# Wrong type entirely -> raises.
try:
    pv.to_series([1, 2, 3])
except TypeError as exc:
    print(f"Expected TypeError (list is not Series/DataFrame): {exc}")

Series input -> passthrough, unchanged.
Single-column DataFrame -> Series: Series AAPL
Multi-column DataFrame + column='MSFT' -> matches master['MSFT']: True
Expected ValueError (multi-column, no column specified): master is a multi-column DataFrame; pass a Series or specify `column=`.
Expected TypeError (list is not Series/DataFrame): data must be a pandas Series or DataFrame, got list.


### 6.5 `align_pair`

In [20]:
r_aapl = master["AAPL"].pct_change().dropna()
r_msft = master["MSFT"].pct_change().dropna()

aligned_a, aligned_b = pv.align_pair(r_aapl, r_msft, "AAPL", "MSFT")
print(f"align_pair: {len(aligned_a)} overlapping observations (inner join, NaNs dropped)")

# No overlap at all -> raises.
disjoint = pd.Series([0.01, 0.02], index=pd.date_range("1990-01-01", periods=2))
try:
    pv.align_pair(r_aapl, disjoint, "AAPL", "disjoint")
except ValueError as exc:
    print(f"Expected ValueError (no overlapping dates): {exc}")

align_pair: 1095 overlapping observations (inner join, NaNs dropped)
Expected ValueError (no overlapping dates): AAPL and disjoint have no overlapping, non-NaN dates. Check that both series share the same date range and calendar.


### 6.6 `periodic_rate_from_annual`

In [21]:
annual = 0.05
for ppy, label in [(TRADING_DAYS_PER_YEAR, "daily(252)"), (CALENDAR_DAYS_PER_YEAR, "daily(365)"),
                    (WEEKS_PER_YEAR, "weekly"), (MONTHS_PER_YEAR, "monthly"), (QUARTERS_PER_YEAR, "quarterly")]:
    periodic = pv.periodic_rate_from_annual(annual, ppy)
    # Round-trip check: compounding the periodic rate ppy times recovers the annual rate.
    roundtrip = (1.0 + periodic) ** ppy - 1.0
    print(f"  5% annual -> {label:12s} periodic rate = {periodic:.6%}  (round-trips to {roundtrip:.4%})")
    assert np.isclose(roundtrip, annual)

  5% annual -> daily(252)   periodic rate = 0.019363%  (round-trips to 5.0000%)
  5% annual -> daily(365)   periodic rate = 0.013368%  (round-trips to 5.0000%)
  5% annual -> weekly       periodic rate = 0.093871%  (round-trips to 5.0000%)
  5% annual -> monthly      periodic rate = 0.407412%  (round-trips to 5.0000%)
  5% annual -> quarterly    periodic rate = 1.227223%  (round-trips to 5.0000%)


### 6.7 `safe_divide`

In [22]:
print("safe_divide(10, 4)  ->", pv.safe_divide(10, 4))
print("safe_divide(5, 0)   ->", pv.safe_divide(5, 0), "(positive numerator / zero -> +inf)")
print("safe_divide(-5, 0)  ->", pv.safe_divide(-5, 0), "(negative numerator / zero -> -inf)")
print("safe_divide(0, 0)   ->", pv.safe_divide(0, 0), "(0/0 -> 0.0, not NaN)")

safe_divide(10, 4)  -> 2.5
safe_divide(5, 0)   -> inf (positive numerator / zero -> +inf)
safe_divide(-5, 0)  -> -inf (negative numerator / zero -> -inf)
safe_divide(0, 0)   -> 0.0 (0/0 -> 0.0, not NaN)


## 7. The `Portfolio` Class

### 7.1 Construction from prices, with weights + asset classes + custom frequency/rf

In [23]:
portfolio = Portfolio(
    master,
    input_type="prices",
    weights=target_weights,
    name="All-Weather Multi-Asset Portfolio",
    frequency=CALENDAR_DAYS_PER_YEAR,  # 365: crypto trades every day of the week
    risk_free_rate=risk_free_rate,
    asset_classes=asset_classes,
)
print(portfolio)
print("weights:\n", portfolio.weights)
print("asset_classes:", portfolio.asset_classes)

Portfolio(name='All-Weather Multi-Asset Portfolio', assets=9, n_obs=1096, frequency=365)
weights:
 AAPL   0.1500
GLD    0.1000
JPM    0.1000
MSFT   0.1300
TLT    0.1000
VNQ    0.1000
BTC    0.1500
ETH    0.1000
SAP    0.0700
Name: weight, dtype: float64
asset_classes: {'AAPL': <AssetClass.EQUITY: 'equity'>, 'MSFT': <AssetClass.EQUITY: 'equity'>, 'JPM': <AssetClass.EQUITY: 'equity'>, 'SAP': <AssetClass.EQUITY: 'equity'>, 'VNQ': <AssetClass.REAL_ESTATE: 'real_estate'>, 'TLT': <AssetClass.BOND: 'bond'>, 'GLD': <AssetClass.COMMODITY: 'commodity'>, 'BTC': <AssetClass.CRYPTO: 'crypto'>, 'ETH': <AssetClass.CRYPTO: 'crypto'>}


### 7.2 Construction errors (bad `data`, bad `input_type`, mismatched `asset_classes`)

In [24]:
try:
    Portfolio(np.array([[1, 2], [3, 4]]))
except TypeError as exc:
    print(f"Expected TypeError (not a DataFrame): {exc}")

try:
    Portfolio(master, input_type="yields")
except ValueError as exc:
    print(f"Expected ValueError (bad input_type): {exc}")

try:
    Portfolio(master, weights=target_weights, asset_classes={"NOT_AN_ASSET": AssetClass.OTHER})
except ValueError as exc:
    print(f"Expected ValueError (asset_classes references unknown asset): {exc}")

try:
    Portfolio(pd.DataFrame({"only_col": [1.0]}, index=pd.DatetimeIndex(["2024-01-01"])))
except ValueError as exc:
    print(f"Expected ValueError (fewer than 2 observations): {exc}")

Expected TypeError (not a DataFrame): Portfolio expects a pandas DataFrame (columns = asset symbols), got ndarray.
Expected ValueError (bad input_type): input_type must be one of ('prices', 'returns'), got 'yields'.
Expected ValueError (asset_classes references unknown asset): asset_classes references unknown assets: ['NOT_AN_ASSET']
Expected ValueError (fewer than 2 observations): data needs at least 2 observation(s), got 1.


### 7.3 Construction from `input_type="returns"` -- equivalent to constructing from prices

In [25]:
asset_returns_df = portfolio.asset_returns()
portfolio_from_returns = Portfolio(
    asset_returns_df,
    input_type="returns",
    weights=target_weights,
    name="Rebuilt from returns",
    frequency=CALENDAR_DAYS_PER_YEAR,
    risk_free_rate=risk_free_rate,
    asset_classes=asset_classes,
)
same_returns = np.allclose(portfolio.returns().to_numpy(), portfolio_from_returns.returns().to_numpy())
print(f"Portfolio(prices=...) and Portfolio(returns=...) produce identical .returns(): {same_returns}")
print(f"(prices() differ in absolute level, since input_type='returns' anchors at 1.0 via prices_from_returns)")
print("original prices tail:\n", portfolio.prices[["AAPL"]].tail(2))
print("reconstructed prices tail:\n", portfolio_from_returns.prices[["AAPL"]].tail(2))

Portfolio(prices=...) and Portfolio(returns=...) produce identical .returns(): True
(prices() differ in absolute level, since input_type='returns' anchors at 1.0 via prices_from_returns)
original prices tail:
                AAPL
Date               
2026-08-05 311.0000
2026-08-06 311.0000
reconstructed prices tail:
              AAPL
2026-08-05 1.7633
2026-08-06 1.7633


### 7.4 Basic accessors: `.prices`, `.asset_names`, `.num_assets`, `.weights`

In [26]:
print("asset_names:", portfolio.asset_names)
print("num_assets: ", portfolio.num_assets)
print("prices.shape:", portfolio.prices.shape)
print("prices is a copy (mutating it doesn't affect the Portfolio):")
p_copy = portfolio.prices
p_copy.iloc[0, 0] = -999.0
print("  mutated copy unaffected internal state:", portfolio.prices.iloc[0, 0] != -999.0)
print("weights:\n", portfolio.weights)

asset_names: ['AAPL', 'GLD', 'JPM', 'MSFT', 'TLT', 'VNQ', 'BTC', 'ETH', 'SAP']
num_assets:  9
prices.shape: (1096, 9)
prices is a copy (mutating it doesn't affect the Portfolio):
  mutated copy unaffected internal state: True
weights:
 AAPL   0.1500
GLD    0.1000
JPM    0.1000
MSFT   0.1300
TLT    0.1000
VNQ    0.1000
BTC    0.1500
ETH    0.1000
SAP    0.0700
Name: weight, dtype: float64


### 7.5 `set_weights` and `set_risk_free_rate`

In [27]:
demo_portfolio = Portfolio(master, weights=target_weights, name="Mutation sandbox", asset_classes=asset_classes)
print("original weights:", demo_portfolio.weights.round(3).to_dict())

demo_portfolio.set_weights({k: 1.0 for k in demo_portfolio.asset_names})  # unnormalized -> re-normalized to equal-weight
print("after set_weights(all 1.0) ->", demo_portfolio.weights.round(3).to_dict())

print("\noriginal risk_free_rate:", demo_portfolio.risk_free_rate)
demo_portfolio.set_risk_free_rate(0.03)
print("after set_risk_free_rate(0.03):", demo_portfolio.risk_free_rate)

original weights: {'AAPL': 0.15, 'GLD': 0.1, 'JPM': 0.1, 'MSFT': 0.13, 'TLT': 0.1, 'VNQ': 0.1, 'BTC': 0.15, 'ETH': 0.1, 'SAP': 0.07}
after set_weights(all 1.0) -> {'AAPL': 0.111, 'GLD': 0.111, 'JPM': 0.111, 'MSFT': 0.111, 'TLT': 0.111, 'VNQ': 0.111, 'BTC': 0.111, 'ETH': 0.111, 'SAP': 0.111}

original risk_free_rate: 0.0
after set_risk_free_rate(0.03): 0.03


### 7.6 `asset_returns` (simple vs. log)

In [28]:
simple_asset_r = portfolio.asset_returns(log=False)
log_asset_r = portfolio.asset_returns(log=True)
print("asset_returns(log=False).tail(2):\n", simple_asset_r.tail(2))
print("\nasset_returns(log=True).tail(2):\n", log_asset_r.tail(2))
print(f"\nshapes match: {simple_asset_r.shape == log_asset_r.shape}")

asset_returns(log=False).tail(2):
              AAPL    GLD    JPM    MSFT    TLT    VNQ    BTC    ETH    SAP
Date                                                                      
2026-08-05 0.0052 0.0414 0.0048 -0.0109 0.0022 0.0000 0.0000 0.0000 0.0000
2026-08-06 0.0000 0.0000 0.0000  0.0000 0.0000 0.0000 0.0066 0.0142 0.0000

asset_returns(log=True).tail(2):
              AAPL    GLD    JPM    MSFT    TLT    VNQ    BTC    ETH    SAP
Date                                                                      
2026-08-05 0.0052 0.0405 0.0048 -0.0109 0.0022 0.0000 0.0000 0.0000 0.0000
2026-08-06 0.0000 0.0000 0.0000  0.0000 0.0000 0.0000 0.0065 0.0141 0.0000

shapes match: True


### 7.7 `returns` -- simple, log, and block-compounded

In [29]:
r_simple = portfolio.returns()
r_log = portfolio.returns(log=True)
r_monthly_blocks = portfolio.returns(period=21)  # ~monthly blocks on a daily series

print(f"returns(): {len(r_simple)} obs, name={r_simple.name!r}")
print(r_simple.tail(3))
print(f"\nreturns(log=True) tail:\n{r_log.tail(3)}")
print(f"\nreturns(period=21) (~monthly compounded blocks): {len(r_monthly_blocks)} obs")
print(r_monthly_blocks.tail(3))

returns(): 1095 obs, name='All-Weather Multi-Asset Portfolio'
Date
2026-08-04   0.0099
2026-08-05   0.0042
2026-08-06   0.0024
Name: All-Weather Multi-Asset Portfolio, dtype: float64

returns(log=True) tail:
Date
2026-08-04   0.0098
2026-08-05   0.0042
2026-08-06   0.0024
Name: All-Weather Multi-Asset Portfolio, dtype: float64

returns(period=21) (~monthly compounded blocks): 53 obs
2026-07-13   0.0193
2026-08-03   0.0528
2026-08-06   0.0166
Name: All-Weather Multi-Asset Portfolio, dtype: float64


### 7.8 `price_index`

In [30]:
idx100 = portfolio.price_index()
idx1000 = portfolio.price_index(base=1000.0)
print(f"price_index(base=100): starts at {idx100.iloc[0]:.2f}, ends at {idx100.iloc[-1]:.2f}")
print(f"price_index(base=1000): starts at {idx1000.iloc[0]:.2f}, ends at {idx1000.iloc[-1]:.2f}")
assert np.isclose(idx1000.iloc[-1] / idx100.iloc[-1], 10.0)

price_index(base=100): starts at 100.00, ends at 183.72
price_index(base=1000): starts at 1000.00, ends at 1837.23


### 7.9 `__repr__`

In [31]:
print(repr(portfolio))

Portfolio(name='All-Weather Multi-Asset Portfolio', assets=9, n_obs=1096, frequency=365)


## 8. Metrics -- Returns (`portpy.metrics.returns`)

### 8.1 `simple_returns`

In [32]:
from portpy.metrics.returns import (
    active_returns, annualized_return, average_return, cagr, cumulative_returns,
    excess_returns, log_returns, prices_from_returns, rebased_returns, simple_returns, total_return,
)

sr_series = simple_returns(master["AAPL"])
sr_frame = simple_returns(master)

print("=" * 50)
print(sr_frame.tail(5))
print("=" * 50)

print(f"simple_returns(Series): {len(sr_series)} obs, matches manual pct_change: "
      f"{np.allclose(sr_series.to_numpy(), master['AAPL'].pct_change().dropna().to_numpy())}")
print(f"simple_returns(DataFrame): shape={sr_frame.shape}")
pexplain("simple_returns")
note("simple_returns")
note_explained("simple_returns")

              AAPL    GLD    JPM    MSFT    TLT     VNQ     BTC     ETH    SAP
Date                                                                          
2026-08-02  0.0000 0.0000 0.0000  0.0000 0.0000  0.0000  0.0115  0.0212 0.0000
2026-08-03 -0.0178 0.0005 0.0024  0.0493 0.0033  0.0012 -0.0003 -0.0129 0.0460
2026-08-04  0.0196 0.0066 0.0138  0.0106 0.0077 -0.0015  0.0094  0.0055 0.0134
2026-08-05  0.0052 0.0414 0.0048 -0.0109 0.0022  0.0000  0.0000  0.0000 0.0000
2026-08-06  0.0000 0.0000 0.0000  0.0000 0.0000  0.0000  0.0066  0.0142 0.0000
simple_returns(Series): 1095 obs, matches manual pct_change: True
simple_returns(DataFrame): shape=(1095, 9)
simple_returns (metric)

What it is:
  The arithmetic percentage change in price from one period to the next. This is the standard return representation used for portfolio calculations.

Formula:
  P_t / P_(t-1) - 1

How to read it:
  A value of 0.01 means the asset gained 1% during that period. A value of -0.02 means it lost 2%.

Good 

### 8.2 `log_returns`

In [33]:
lr_series = log_returns(master["AAPL"])
lr_frame = log_returns(master)
manual_log = np.log(master["AAPL"] / master["AAPL"].shift(1)).iloc[1:]

print("=" * 50)
print(lr_frame.tail(5))
print("=" * 50)

print(f"log_returns(Series): {len(lr_series)} obs, matches manual calc: {np.allclose(lr_series, manual_log)}")
print(f"log_returns(DataFrame): shape={lr_frame.shape}")
pexplain("log_returns")
note("log_returns")
note_explained("log_returns")

              AAPL    GLD    JPM    MSFT    TLT     VNQ     BTC     ETH    SAP
Date                                                                          
2026-08-02  0.0000 0.0000 0.0000  0.0000 0.0000  0.0000  0.0114  0.0210 0.0000
2026-08-03 -0.0179 0.0005 0.0024  0.0482 0.0033  0.0012 -0.0003 -0.0130 0.0450
2026-08-04  0.0195 0.0066 0.0137  0.0105 0.0076 -0.0015  0.0093  0.0054 0.0133
2026-08-05  0.0052 0.0405 0.0048 -0.0109 0.0022  0.0000  0.0000  0.0000 0.0000
2026-08-06  0.0000 0.0000 0.0000  0.0000 0.0000  0.0000  0.0065  0.0141 0.0000
log_returns(Series): 1095 obs, matches manual calc: True
log_returns(DataFrame): shape=(1095, 9)
log_returns (metric)

What it is:
  The continuously compounded return calculated from the logarithmic change in price. Commonly used in quantitative finance and statistical modeling.

Formula:
  ln(P_t / P_(t-1))

How to read it:
  A value of 0.02 represents approximately a 2% continuously compounded return for the period.

Good vs. bad:
  Useful 

### 8.3 `cumulative_returns`

In [34]:
port_r = portfolio.returns()
cum = cumulative_returns(port_r)
print(f"cumulative_returns: starts at {cum.iloc[0]:+.4%}, ends at {cum.iloc[-1]:+.2%}")
print(cum.tail(3))
pexplain("cumulative_returns")
note("cumulative_returns")
note_explained("cumulative_returns")

cumulative_returns: starts at +0.2747%, ends at +83.72%
Date
2026-08-04   0.8251
2026-08-05   0.8328
2026-08-06   0.8372
Name: All-Weather Multi-Asset Portfolio, dtype: float64
cumulative_returns (metric)

What it is:
  The compounded growth of an investment over a sequence of returns, showing how wealth evolves through time.

Formula:
  prod(1 + r_t) - 1

How to read it:
  A value of 0.25 means an initial investment increased by 25% over the entire measured period.

Good vs. bad:
  Higher cumulative return indicates stronger absolute performance, but it must be evaluated alongside volatility, drawdown, and investment horizon.

Caveats:
  Cumulative return ignores the path taken. Two investments can have the same ending return but very different risk experiences.


### 8.4 `prices_from_returns` (and its exact round-trip with `simple_returns`)

In [35]:
reconstructed = prices_from_returns(port_r, base=100.0)
roundtrip_returns = simple_returns(reconstructed)
roundtrip_ok = np.allclose(roundtrip_returns.to_numpy(), port_r.to_numpy())

print("=" * 50)
print(roundtrip_returns.tail(5))
print("=" * 50)

print(f"prices_from_returns(base=100): {len(reconstructed)} obs (1 more than input -- anchor row)")
print(f"simple_returns(prices_from_returns(r)) round-trips r exactly: {roundtrip_ok}")

# DataFrame form too.
reconstructed_frame = prices_from_returns(simple_returns(master), base=1.0)
print(f"prices_from_returns on a DataFrame: shape={reconstructed_frame.shape}")
pexplain("prices_from_returns")
note("prices_from_returns")
note_explained("prices_from_returns")

2026-08-02   0.0038
2026-08-03   0.0064
2026-08-04   0.0099
2026-08-05   0.0042
2026-08-06   0.0024
Name: All-Weather Multi-Asset Portfolio, dtype: float64
prices_from_returns(base=100): 1096 obs (1 more than input -- anchor row)
simple_returns(prices_from_returns(r)) round-trips r exactly: True
prices_from_returns on a DataFrame: shape=(1096, 9)
prices_from_returns (metric)

What it is:
  Transforms a return series into a synthetic price/equity curve by compounding returns from a chosen starting value.

Formula:
  base * prod(1 + r_t)

How to read it:
  A resulting value of 1.20 means a starting investment of 1.00 grew to 1.20 after applying the return sequence.

Good vs. bad:
  Useful for creating equity curves from return data and enabling price-based analysis such as drawdowns and CAGR.

Caveats:
  This does not recreate real market prices. It only reconstructs the growth path implied by the returns.


### 8.5 `total_return`

In [36]:
tr = total_return(port_r)
tr_result = total_return(port_r, as_result=True)
print(f"total_return: {tr:+.2%}  |  as_result=True -> {tr_result!r}")
tr_result.explain()
note("total_return")
note_explained("total_return")

total_return: +83.72%  |  as_result=True -> total_return=0.837233 % - +83.7% total return
total_return (metric)

What it is:
  The total compounded gain or loss achieved over the entire investment period without converting it into an annual rate.

Formula:
  prod(1 + r_t) - 1

How to read it:
  A value of 0.35 means an investment gained 35% from start to finish.

Good vs. bad:
  Useful for measuring absolute performance, but comparisons should use the same time period and similar risk exposure.

Caveats:
  Not annualized. The same total return can represent very different performance depending on whether it occurred over months or years.

This result:
  +83.7% total return


### 8.6 `annualized_return` (geometric vs. arithmetic)

In [37]:
geo = annualized_return(port_r, periods_per_year=portfolio.frequency, geometric=True)
arith = annualized_return(port_r, periods_per_year=portfolio.frequency, geometric=False)
print(f"annualized_return: geometric={geo:+.2%}, arithmetic={arith:+.2%}")
annualized_return(port_r, periods_per_year=portfolio.frequency, as_result=True).explain()
note("annualized_return")
note_explained("annualized_return")

annualized_return: geometric=+22.48%, arithmetic=+21.92%
annualized_return (metric)

What it is:
  The return converted into an annual growth rate, allowing comparison between investments with different measurement periods.

Formula:
  geometric: prod(1+r)^(periods_per_year/n) - 1 | arithmetic: mean(r) * periods_per_year

How to read it:
  A value of 0.12 means the investment produced an equivalent annualized return of 12%.

Good vs. bad:
  Higher annualized returns are generally preferable, but they should always be evaluated together with volatility, drawdown, and consistency.

Caveats:
  Geometric annualization is preferred because it accounts for compounding. Arithmetic annualization can overstate expected growth in volatile series.

This result:
  +22.5% annualized return


### 8.7 `cagr`

In [38]:
cagr_result = cagr(portfolio.price_index(), periods_per_year=portfolio.frequency, as_result=True)
print(cagr_result)
print(repr(cagr_result))
cagr_result.explain()
note("cagr")
note_explained("cagr")

0.2247705621245748
cagr=0.224771 % - +22.5%/year compounded
cagr (metric)

What it is:
  The constant annual growth rate required for an investment to move from its starting value to its ending value.

Formula:
  (P_end / P_start)^(1/years) - 1

How to read it:
  A CAGR of 0.10 means the investment grew as if it compounded at 10% per year.

Good vs. bad:
  Higher CAGR indicates stronger long-term growth, but it does not describe volatility or the path taken.

Caveats:
  CAGR ignores intermediate fluctuations. Two investments with identical CAGR can have completely different risk profiles.

This result:
  +22.5%/year compounded


### 8.8 `average_return` (arithmetic vs. geometric)

In [39]:
avg_arith = average_return(port_r, geometric=False)
avg_geo = average_return(port_r, geometric=True)
print(f"average_return: arithmetic={avg_arith:+.4%}/period, geometric={avg_geo:+.4%}/period")
average_return(port_r, as_result=True).explain()
note("average_return")
note_explained("average_return")

average_return: arithmetic=+0.0601%/period, geometric=+0.0556%/period
average_return (metric)

What it is:
  The average return per observation, calculated either arithmetically or geometrically depending on the selected method.

Formula:
  arithmetic: mean(r) | geometric: exp(mean(ln(1+r))) - 1

How to read it:
  This is the average return per period (for example daily), not an annual performance measure.

Good vs. bad:
  Useful for understanding return behavior, but should not be used alone because it ignores dispersion and downside risk.

Caveats:
  Arithmetic averages can overstate realized growth when returns are volatile. Geometric averages better represent compounded wealth growth.

This result:
  +0.060% average period return


### 8.9 `rebased_returns`

In [40]:
rebased = rebased_returns(portfolio.price_index(), base=100.0)
print(f"rebased_returns: starts at {rebased.iloc[0]:.2f}, ends at {rebased.iloc[-1]:.2f}")
rebased_50 = rebased_returns(portfolio.price_index(), base=50.0)
print(f"rebased_returns(base=50): starts at {rebased_50.iloc[0]:.2f}")
pexplain("rebased_returns")
note("rebased_returns")
note_explained("rebased_returns")

rebased_returns: starts at 100.00, ends at 183.72
rebased_returns(base=50): starts at 50.00
rebased_returns (metric)

What it is:
  Rescales a price series to a common starting value while preserving all relative price movements.

Formula:
  price / first_price * base

How to read it:
  With a base of 100, a value of 150 means the investment increased by 50% from the starting point.

Good vs. bad:
  Useful for visual comparison of assets with different price levels.

Caveats:
  Rebasing changes only the displayed scale. It does not change returns, risk, or performance statistics.


### 8.10 `excess_returns` (vs. a flat rate, and vs. a benchmark series)

In [41]:
benchmark_returns = benchmark_prices.pct_change().dropna()

excess_vs_flat = excess_returns(port_r, 0.0001)
excess_vs_bench = excess_returns(port_r, benchmark_returns)
print(f"excess_returns(vs 1bp/period flat rate): mean={excess_vs_flat.mean():+.5f}")
print(f"excess_returns(vs SPY): mean={excess_vs_bench.mean():+.5f}, n={len(excess_vs_bench)}")
pexplain("excess_returns")
note("excess_returns")
note_explained("excess_returns")

excess_returns(vs 1bp/period flat rate): mean=+0.00050
excess_returns(vs SPY): mean=-0.00006, n=751
excess_returns (metric)

What it is:
  The return earned above a benchmark return or risk-free rate during the same period.

Formula:
  portfolio return - benchmark or risk-free return

How to read it:
  A value of 0.03 means the portfolio outperformed the reference by 3% during that period.

Good vs. bad:
  Positive excess return indicates outperformance relative to the chosen reference. Negative values indicate underperformance.

Caveats:
  Excess return does not account for risk taken. A higher excess return may simply come from accepting higher volatility.


### 8.11 `active_returns` (alias of `excess_returns` restricted to a benchmark series)

In [42]:
active = active_returns(port_r, benchmark_returns)
print(f"active_returns == excess_returns(vs benchmark): {active.equals(excess_vs_bench)}")
pexplain("active_returns")
note("active_returns")
note_explained("active_returns")

active_returns == excess_returns(vs benchmark): True
active_returns (metric)

What it is:
  The return difference between a portfolio and its benchmark, representing the performance generated by active decisions.

Formula:
  portfolio return - benchmark return

How to read it:
  A value of 0.01 means the portfolio exceeded the benchmark by 1% during that period.

Good vs. bad:
  Positive active returns indicate benchmark outperformance. They should be evaluated with tracking error and information ratio.

Caveats:
  Active return alone does not measure consistency. A portfolio can have high active returns but poor risk-adjusted performance.


## 9. Metrics -- Risk (`portpy.metrics.risk`)

### 9.1 `variance`

In [43]:
from portpy.metrics.risk import (
    beta, conditional_var, downside_deviation, kurtosis, pain_index, semi_variance,
    skewness, tail_ratio, tracking_error, ulcer_index, value_at_risk, variance, volatility,
)

var_result = portfolio.metrics.variance(as_result=True)
print(var_result, "|", repr(var_result))
var_result.explain()
note("variance")
note_explained("variance")

9.027287333615358e-05 | variance=9.02729e-05 - 0.000090 (in squared-return units; sqrt gives volatility = 0.95%)
variance (metric)

What it is:
  The average squared deviation of returns from their mean - the textbook measure of dispersion.

Formula:
  mean((r - mean(r))^2), ddof=1

How to read it:
  In squared-return units, so it's hard to interpret directly - use volatility (its square root) instead for anything intuitive.

Good vs. bad:
  Lower means more consistent returns; there's no universal good/bad cutoff on its own.

This result:
  0.000090 (in squared-return units; sqrt gives volatility = 0.95%)


### 9.2 `volatility` (annualized vs. raw)

In [44]:
vol_ann = portfolio.metrics.volatility(annualized=True)
vol_raw = portfolio.metrics.volatility(annualized=False)
print(f"volatility(annualized=True)={vol_ann:.2%}, volatility(annualized=False)={vol_raw:.4%}")
portfolio.metrics.volatility(as_result=True).explain()
note("volatility")
note_explained("volatility")

volatility(annualized=True)=18.15%, volatility(annualized=False)=0.9501%
volatility (metric)

What it is:
  The standard deviation of returns - the most common measure of how much an asset's returns bounce around.

Formula:
  std(r, ddof=1) * sqrt(periods_per_year)  [if annualized]

How to read it:
  Annualized volatility of 0.20 means returns typically swing about +/-20%/year around the average.

Good vs. bad:
  Lower is 'safer' in the sense of smoother returns, but volatility alone says nothing about direction - a fast-rising asset can be just as volatile as a falling one. Always read it alongside the return.

Caveats:
  Penalizes upside and downside swings equally, unlike downside_deviation or semi_variance.

This result:
  18.2%/yr (moderate)


### 9.3 `downside_deviation` (varying `mar`)

In [45]:
for mar in (0.0, 0.02, 0.05):
    dd = portfolio.metrics.downside_deviation(mar=mar)
    print(f"downside_deviation(mar={mar:.0%}) = {dd:.2%}")
portfolio.metrics.downside_deviation(as_result=True).explain()
note("downside_deviation")
note_explained("downside_deviation")

downside_deviation(mar=0%) = 12.14%
downside_deviation(mar=2%) = 12.19%
downside_deviation(mar=5%) = 12.26%
downside_deviation (metric)

What it is:
  Like volatility, but only counts returns that fall short of a minimum acceptable return (MAR) - upside swings aren't penalized.

Formula:
  sqrt(mean(min(r - mar, 0)^2)) * sqrt(periods_per_year)  [if annualized]

How to read it:
  Same units and scale as volatility, but will always be <= volatility for the same series since only bad periods count.

Good vs. bad:
  Lower is better. Compare it to volatility: a downside deviation much smaller than volatility means most of the swings are on the upside.

Caveats:
  Divides by the full sample size N (Sortino's original convention) - see semi_variance for the alternative that divides only by below-threshold periods.

This result:
  12.1%/yr of downside risk


### 9.4 `semi_variance` (varying `mar`)

In [46]:
for mar in (0.0, 0.02):
    sv = portfolio.metrics.semi_variance(mar=mar)
    print(f"semi_variance(mar={mar:.0%}) = {sv:.10f}")
portfolio.metrics.semi_variance(as_result=True).explain()
note("semi_variance")
note_explained("semi_variance")

semi_variance(mar=0%) = 0.0000900047
semi_variance(mar=2%) = 0.0000896324
semi_variance (metric)

What it is:
  Mean squared shortfall below a minimum acceptable return (MAR), averaged only over the periods that actually fell short.

Formula:
  mean((r - mar)^2 for r < mar)

How to read it:
  In squared-return units; compare across portfolios rather than reading in isolation.

Good vs. bad:
  Lower is better (less/milder downside). More sensitive to a few very bad periods than downside_deviation, since it doesn't dilute by the full sample size.

Caveats:
  If nothing fell below the MAR, this is exactly 0 - always check how many periods were below threshold before trusting a 0.

This result:
  0.000090 squared-return units below the MAR


### 9.5 `value_at_risk` -- all 3 methods x 2 confidence levels

In [47]:
for method in ("historical", "parametric", "cornish_fisher"):
    for confidence in (0.95, 0.99):
        v = portfolio.metrics.value_at_risk(method=method, confidence=confidence)
        print(f"value_at_risk[{method:15s}, {confidence:.0%}] = {v:+.2%}")

try:
    portfolio.metrics.value_at_risk(method="monte_carlo")
except ValueError as exc:
    print(f"\nExpected ValueError (unknown method): {exc}")
portfolio.metrics.value_at_risk(as_result=True).explain()
note("value_at_risk")
note_explained("value_at_risk")

value_at_risk[historical     , 95%] = -1.50%
value_at_risk[historical     , 99%] = -2.58%
value_at_risk[parametric     , 95%] = -1.50%
value_at_risk[parametric     , 99%] = -2.15%
value_at_risk[cornish_fisher , 95%] = -1.29%
value_at_risk[cornish_fisher , 99%] = -3.20%

Expected ValueError (unknown method): method must be one of ('historical', 'parametric', 'cornish_fisher'), got 'monte_carlo'.
value_at_risk (metric)

What it is:
  The loss threshold you'd only expect to breach a small fraction of the time (e.g. 5% of periods at 95% confidence).

Formula:
  historical: percentile(r, 100*(1-confidence))  |  parametric: mean + z*std

How to read it:
  A 95% VaR of -0.03 means: in 95% of periods, you did NOT lose more than 3%. It does NOT say anything about how bad the worst 5% get.

Good vs. bad:
  Closer to zero (less negative) is better/safer. Only comparable across series measured at the same confidence level and frequency.

Caveats:
  VaR ignores everything beyond the threshold - two

### 9.6 `conditional_var`

In [48]:
for confidence in (0.95, 0.99):
    cvar = portfolio.metrics.conditional_var(confidence=confidence)
    var95 = portfolio.metrics.value_at_risk(confidence=confidence)
    print(f"conditional_var[{confidence:.0%}]={cvar:+.2%}  (<= value_at_risk={var95:+.2%}: {cvar <= var95})")
portfolio.metrics.conditional_var(as_result=True).explain()
note("conditional_var")
note_explained("conditional_var")

conditional_var[95%]=-2.15%  (<= value_at_risk=-1.50%: True)
conditional_var[99%]=-3.25%  (<= value_at_risk=-2.58%: True)
conditional_var (metric)

What it is:
  Also called Expected Shortfall (ES/CVaR): the average loss *given* that you're already in the bad tail defined by VaR.

Formula:
  mean(r for r <= VaR cutoff)

How to read it:
  A 95% CVaR of -0.05 means: on the worst 5% of periods, the average loss was 5%.

Good vs. bad:
  Closer to zero is better/safer. Always at least as negative as VaR at the same confidence - a large gap between VaR and CVaR signals a fat, dangerous tail.

Caveats:
  Still a historical/empirical estimate - a tail event worse than anything in the sample isn't reflected.

This result:
  -2.15% average loss in the worst-case tail


### 9.7 `tail_ratio`

In [49]:
print(f"tail_ratio = {portfolio.metrics.tail_ratio():.2f}")
portfolio.metrics.tail_ratio(as_result=True).explain()
note("tail_ratio")
note_explained("tail_ratio")

tail_ratio = 1.02
tail_ratio (metric)

What it is:
  Compares the size of big gains (95th percentile) to big losses (5th percentile).

Formula:
  |percentile(r, 95)| / |percentile(r, 5)|

How to read it:
  A ratio of 1.0 means big gains and big losses are similarly sized; below 1.0 means losses in the tail are bigger than gains.

Good vs. bad:
  Above 1.0 is generally favorable (upside tail bigger than downside tail); well below 1.0 (e.g. 0.25) signals a strategy prone to occasional large losses.

This result:
  1.02 (roughly symmetric tails)


### 9.8 `skewness`

In [50]:
print(f"skewness = {portfolio.metrics.skewness():+.3f}")
portfolio.metrics.skewness(as_result=True).explain()
note("skewness")
note_explained("skewness")

skewness = +0.359
skewness (metric)

What it is:
  Measures asymmetry in the return distribution: whether extreme moves tend to be on the upside or downside.

Formula:
  adjusted Fisher-Pearson sample skewness

How to read it:
  Positive skew = occasional large gains with frequent small losses; negative skew = occasional large losses with frequent small gains.

Good vs. bad:
  Positive is generally considered more desirable for a long-only investor (limited, frequent losses; occasional big wins), though many trend/momentum strategies deliberately run negative skew in exchange for a higher hit rate.

Caveats:
  Very sensitive to sample size and outliers in short return histories.

This result:
  +0.36 (roughly symmetric)


### 9.9 `kurtosis`

In [51]:
print(f"kurtosis (excess) = {portfolio.metrics.kurtosis():+.3f}")
portfolio.metrics.kurtosis(as_result=True).explain()
note("kurtosis")
note_explained("kurtosis")

kurtosis (excess) = +6.110
kurtosis (metric)

What it is:
  Measures how fat the tails of the return distribution are relative to a Normal distribution.

Formula:
  excess kurtosis = kurtosis - 3 (so 0 = Normal-like)

How to read it:
  Positive excess kurtosis means extreme returns (both directions) are more frequent than a Normal distribution would predict.

Good vs. bad:
  There's no 'good' side - high kurtosis just means fatter tails, i.e. VaR/volatility computed assuming Normality will understate real tail risk.

Caveats:
  Financial returns are almost always leptokurtic (excess kurtosis > 0) - don't be alarmed by a positive number, but do combine with cornish_fisher VaR instead of parametric VaR when it's large.

This result:
  +6.11 (fat tails - Normal-based risk estimates will understate real risk)


### 9.10 `ulcer_index`

In [52]:
ui = ulcer_index(portfolio.price_index())
print(f"ulcer_index = {ui:.2%}")
ulcer_index(portfolio.price_index(), as_result=True).explain()
note("ulcer_index")
note_explained("ulcer_index")

ulcer_index = 7.98%
ulcer_index (metric)

What it is:
  A drawdown-based risk measure that penalizes both the depth and the duration of drawdowns (unlike volatility, which ignores duration).

Formula:
  sqrt(mean(drawdown_pct^2)) over the whole price history

How to read it:
  Higher values mean drawdowns were deeper and/or the portfolio spent more time underwater.

Good vs. bad:
  Lower is better. Useful for comparing strategies with similar volatility but very different drawdown *experience* (a choppy-but-quick-to-recover series scores lower than a slow bleed of the same depth).

This result:
  7.98%


### 9.11 `pain_index`

In [53]:
pi = pain_index(portfolio.price_index())
print(f"pain_index = {pi:.2%}")
pain_index(portfolio.price_index(), as_result=True).explain()
note("pain_index")
note_explained("pain_index")

pain_index = 5.45%
pain_index (metric)

What it is:
  The average drawdown magnitude across the entire history - literally 'how much pain, on average, was this investor in?'

Formula:
  mean(|drawdown_pct|) over the whole price history

How to read it:
  A pain index of 0.05 means the portfolio was, on average, 5% below its running peak.

Good vs. bad:
  Lower is better. Often paired with return to form a 'pain ratio' analogous to Calmar but using average rather than max drawdown.

This result:
  5.45% average distance below peak


### 9.12 `beta`

In [54]:
port_beta = portfolio.metrics.beta(benchmark=benchmark_returns)
print(f"beta vs SPY = {port_beta:+.3f}")
portfolio.metrics.beta(benchmark=benchmark_returns, as_result=True).explain()
note("beta")
note_explained("beta")

beta vs SPY = +0.807
beta (metric)

What it is:
  How much the portfolio tends to move for every 1-unit move in a benchmark - its sensitivity to market/benchmark risk.

Formula:
  Cov(returns, benchmark) / Var(benchmark)

How to read it:
  Beta of 1.2 means the portfolio historically moved about 20% more than the benchmark, in the same direction, on average.

Good vs. bad:
  Neither high nor low beta is inherently 'good' - it depends on your goal. Beta < 1 suggests a defensive profile; beta > 1 an aggressive one; beta near 0 suggests low correlation to that particular benchmark (not necessarily low risk overall).

Caveats:
  A single historical beta assumes a stable linear relationship - it can shift substantially in market stress (beta instability).

This result:
  0.81 (more defensive than the benchmark)


### 9.13 `tracking_error`

In [55]:
te = portfolio.metrics.tracking_error(benchmark=benchmark_returns)
te_raw = portfolio.metrics.tracking_error(benchmark=benchmark_returns, annualized=False)
print(f"tracking_error(annualized=True)={te:.2%}, tracking_error(annualized=False)={te_raw:.4%}")
portfolio.metrics.tracking_error(benchmark=benchmark_returns, as_result=True).explain()
note("tracking_error")
note_explained("tracking_error")

tracking_error(annualized=True)=15.18%, tracking_error(annualized=False)=0.7944%
tracking_error (metric)

What it is:
  How much the portfolio's returns deviate, period to period, from a benchmark - the volatility of the *difference*, not of the portfolio itself.

Formula:
  std(returns - benchmark, ddof=1) * sqrt(periods_per_year)  [if annualized]

How to read it:
  A tracking error of 0.03/yr means the portfolio's return typically differs from the benchmark's by about 3 percentage points per year.

Good vs. bad:
  For an index-tracking strategy, lower is better (tighter tracking). For an active strategy seeking to beat a benchmark, some tracking error is expected and even desirable - judge it together with information_ratio, not alone.

This result:
  15.18%/yr


## 10. Metrics -- Risk-Adjusted Performance (`portpy.metrics.performance`)

### 10.1 `sharpe_ratio`

In [56]:
sharpe = portfolio.metrics.sharpe_ratio(as_result=True)
sharpe_unannualized = portfolio.metrics.sharpe_ratio(annualized=False)
print(f"{sharpe!r}")
print(f"sharpe_ratio(annualized=False) = {sharpe_unannualized:.4f}")
sharpe.explain()
note("sharpe_ratio")
note_explained("sharpe_ratio")

sharpe_ratio=1.00624 - good
sharpe_ratio(annualized=False) = 0.0527
sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.

Caveats:
  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.

This result:
  good


### 10.2 `sortino_ratio`

In [57]:
for mar in (0.0, 0.03):
    sortino = portfolio.metrics.sortino_ratio(mar=mar, as_result=True)
    print(f"sortino_ratio(mar={mar:.0%}) -> {sortino!r}")
sortino.explain()
note("sortino_ratio")
note_explained("sortino_ratio")

sortino_ratio(mar=0%) -> sortino_ratio=1.80628 - good
sortino_ratio(mar=3%) -> sortino_ratio=1.55342 - good
sortino_ratio (metric)

What it is:
  Measures excess return relative to downside risk only, ignoring upside volatility.

Formula:
  mean(r - mar) / downside_deviation(r, mar) * sqrt(periods_per_year)

How to read it:
  Higher values indicate that returns are being achieved with less harmful downside variation. Compare only when MAR assumptions are consistent.

Good vs. bad:
  Higher is generally better. A large gap between Sortino and Sharpe suggests that volatility is mostly coming from positive returns rather than losses.

Caveats:
  Sensitive to the chosen MAR. Different MAR assumptions can produce significantly different results.

This result:
  good


### 10.3 `calmar_ratio`

In [58]:
calmar = portfolio.metrics.calmar_ratio(as_result=True)
print(repr(calmar))
calmar.explain()
note("calmar_ratio")
note_explained("calmar_ratio")

calmar_ratio=1.02802 - solid
calmar_ratio (metric)

What it is:
  Measures annualized return relative to the portfolio's largest historical drawdown.

Formula:
  annualized_return / abs(max_drawdown)

How to read it:
  A Calmar of 2 means the annualized return is roughly twice the size of the worst observed drawdown.

Good vs. bad:
  Higher values indicate more return generated relative to worst historical loss. Values above 3 are generally considered strong, but depend on the sample period.

Caveats:
  Highly dependent on the worst historical event in the sample. Short histories may underestimate future drawdown risk.

This result:
  solid


### 10.4 `omega_ratio`

In [59]:
for threshold in (0.0, 0.05):
    omega = portfolio.metrics.omega_ratio(threshold=threshold)
    print(f"omega_ratio(threshold={threshold:.0%}) = {omega:.3f}")
portfolio.metrics.omega_ratio(as_result=True).explain()
note("omega_ratio")
note_explained("omega_ratio")

omega_ratio(threshold=0%) = 1.202
omega_ratio(threshold=5%) = 1.154
omega_ratio (metric)

What it is:
  Compares the probability-weighted magnitude of gains and losses relative to a chosen return threshold.

Formula:
  sum(gains above threshold) / sum(abs(losses below threshold))

How to read it:
  A value above 1 means gains above the threshold outweigh losses below it. A value below 1 means losses dominate.

Good vs. bad:
  Higher is better because it captures the full return distribution rather than only mean and volatility.

Caveats:
  Highly dependent on the selected threshold. Results can change substantially when the target return changes.

This result:
  1.20 (gains dominate)


### 10.5 `information_ratio`

In [60]:
ir = portfolio.metrics.information_ratio(benchmark=benchmark_returns, as_result=True)
print(repr(ir))
ir.explain()
note("information_ratio")
note_explained("information_ratio")

information_ratio=-0.1487 - negative active performance
information_ratio (metric)

What it is:
  Measures active return generated over a benchmark relative to the consistency of that outperformance.

Formula:
  mean(returns - benchmark) / std(returns - benchmark, ddof=1) * sqrt(periods_per_year)

How to read it:
  Higher values indicate more consistent benchmark outperformance. A value near zero means active returns are not reliably different from the benchmark.

Good vs. bad:
  Higher is better. Sustained values above 1 are uncommon and indicate strong active management consistency.

Caveats:
  Depends heavily on benchmark selection. A poor benchmark can make the ratio misleading.

This result:
  negative active performance


### 10.6 `treynor_ratio`

In [61]:
treynor = portfolio.metrics.treynor_ratio(beta=port_beta, as_result=True)
print(f"beta used: {port_beta:+.3f}  ->  {treynor!r}")
treynor.explain()
note("treynor_ratio")
note_explained("treynor_ratio")

beta used: +0.807  ->  treynor_ratio=0.226283 - +0.226 excess return per beta unit
treynor_ratio (metric)

What it is:
  Measures excess return earned per unit of systematic market risk measured by beta.

Formula:
  mean(r - rf) / beta * periods_per_year

How to read it:
  Higher values indicate more return generated for each unit of market exposure. It is mainly useful when comparing diversified portfolios with similar benchmarks.

Good vs. bad:
  Higher is better, but only meaningful when beta is stable and accurately estimated.

Caveats:
  Ignores idiosyncratic risk. Results become unstable when beta is close to zero or changes significantly over time.

This result:
  +0.226 excess return per beta unit


### 10.7 `m2_measure`

In [62]:
m2 = portfolio.metrics.m2_measure(benchmark=benchmark_returns, as_result=True)
print(repr(m2))
m2.explain()
note("m2_measure")
note_explained("m2_measure")

m2_measure=0.245687 % - +24.6%/yr risk-adjusted return
m2_measure (metric)

What it is:
  Converts Sharpe ratio into a return percentage by adjusting the portfolio to the benchmark volatility level.

Formula:
  rf + sharpe_ratio(returns) * volatility(benchmark)

How to read it:
  Can be compared directly with benchmark returns. A higher M2 indicates better risk-adjusted performance after matching volatility.

Good vs. bad:
  Higher than the benchmark return indicates superior risk-adjusted performance.

Caveats:
  Assumes volatility is the relevant risk measure and depends on the benchmark used for comparison.

This result:
  +24.6%/yr risk-adjusted return


### 10.8 `sterling_ratio`

In [63]:
for n in (3, 5):
    sterling = portfolio.metrics.sterling_ratio(n=n)
    print(f"sterling_ratio(n={n}) = {sterling:.3f}")
portfolio.metrics.sterling_ratio(as_result=True).explain()
note("sterling_ratio")
note_explained("sterling_ratio")

sterling_ratio(n=3) = 1.281
sterling_ratio(n=5) = 1.671
sterling_ratio (metric)

What it is:
  Measures annualized return relative to the average magnitude of the largest historical drawdowns.

Formula:
  annualized_return / mean(abs(n worst drawdowns))

How to read it:
  Higher values indicate more return generated relative to repeated severe drawdown events.

Good vs. bad:
  Higher is better. It is less dominated by a single worst event than Calmar ratio.

Caveats:
  This implementation uses average historical drawdowns. Some definitions of Sterling ratio use different drawdown adjustments.

This result:
  1.67


### 10.9 `burke_ratio`

In [64]:
for n in (3, 5):
    burke = portfolio.metrics.burke_ratio(n=n)
    print(f"burke_ratio(n={n}) = {burke:.3f}")
portfolio.metrics.burke_ratio(as_result=True).explain()
note("burke_ratio")
note_explained("burke_ratio")

burke_ratio(n=3) = 0.590
burke_ratio(n=5) = 0.561
burke_ratio (metric)

What it is:
  Measures excess return relative to the combined impact of the largest drawdowns, giving more weight to extreme losses.

Formula:
  (annualized_return - rf) / sqrt(sum(worst_drawdowns^2))

How to read it:
  Higher values indicate stronger return generation after accounting for severe drawdown history.

Good vs. bad:
  Higher is better. Lower values indicate that drawdown severity consumes more of the portfolio's return.

Caveats:
  Sensitive to the selected number of drawdowns included and the historical sample.

This result:
  0.56


### 10.10 `gain_to_pain_ratio`

In [65]:
gpr = portfolio.metrics.gain_to_pain_ratio()
print(f"gain_to_pain_ratio = {gpr:.3f}")
portfolio.metrics.gain_to_pain_ratio(as_result=True).explain()
note("gain_to_pain_ratio")
note_explained("gain_to_pain_ratio")

gain_to_pain_ratio = 0.202
gain_to_pain_ratio (metric)

What it is:
  Compares cumulative gains against cumulative losses, focusing on the balance between positive and negative returns.

Formula:
  sum(positive returns) / sum(abs(negative returns))

How to read it:
  Values above 1 indicate that gains outweigh losses. Higher values indicate a more favorable return distribution.

Good vs. bad:
  Higher is better, but it does not account for volatility, drawdown depth, or the path taken to achieve returns.

Caveats:
  Ignores compounding effects and timing of returns. Two strategies with the same ratio may have very different risk profiles.

This result:
  poor


### 10.11 `kappa_three_ratio`

In [66]:
for mar in (0.0, 0.02):
    k3 = portfolio.metrics.kappa_three_ratio(mar=mar)
    print(f"kappa_three_ratio(mar={mar:.0%}) = {k3:.3f}")
portfolio.metrics.kappa_three_ratio(as_result=True).explain()
note("kappa_three_ratio")
note_explained("kappa_three_ratio")

kappa_three_ratio(mar=0%) = 0.065
kappa_three_ratio(mar=2%) = 0.059
kappa_three_ratio (metric)

What it is:
  Measures return relative to downside risk using a third-order penalty for returns below the minimum acceptable return.

Formula:
  mean(r - mar) / mean(max(mar - r, 0)^3)^(1/3)

How to read it:
  Higher values indicate better compensation for severe downside outcomes. It penalizes extreme losses more than Sortino ratio.

Good vs. bad:
  Higher is better when comparing strategies using the same MAR assumption.

Caveats:
  More sensitive to extreme observations than Sortino, which can make it unstable with short samples.

This result:
  0.06


### 10.12 `upside_potential_ratio`

In [67]:
for mar in (0.0, 0.02):
    upr = portfolio.metrics.upside_potential_ratio(mar=mar)
    print(f"upside_potential_ratio(mar={mar:.0%}) = {upr:.3f}")
portfolio.metrics.upside_potential_ratio(as_result=True).explain()
note("upside_potential_ratio")
note_explained("upside_potential_ratio")

upside_potential_ratio(mar=0%) = 0.563
upside_potential_ratio(mar=2%) = 0.556
upside_potential_ratio (metric)

What it is:
  Measures average upside above the MAR relative to downside deviation below the MAR.

Formula:
  mean(max(r - mar, 0)) / downside_deviation(r, mar)

How to read it:
  Higher values indicate that upside potential is large relative to downside risk.

Good vs. bad:
  Higher is better, especially for strategies targeting asymmetric return profiles.

Caveats:
  Depends on MAR selection and does not measure drawdown behavior directly.

This result:
  0.56


## 11. Metrics -- Drawdowns (`portpy.metrics.drawdowns`)

### 11.1 `drawdown_series`

In [68]:
dd = portfolio.metrics.drawdown_series()
print(f"drawdown_series: range [{dd.min():.2%}, {dd.max():.2%}], currently {dd.iloc[-1]:.2%}")
print(dd.tail(3))
pexplain("drawdown_series")
note("drawdown_series")
note_explained("drawdown_series")

drawdown_series: range [-21.86%, 0.00%], currently -11.52%
2026-08-04   -0.1210
2026-08-05   -0.1173
2026-08-06   -0.1152
Name: drawdown, dtype: float64
drawdown_series (metric)

What it is:
  The percentage decline of a portfolio value from its highest previous level at each point in time.

Formula:
  current_value / running_max_value - 1

How to read it:
  Values are always zero or negative. A value of -0.20 means the portfolio is 20% below its previous peak.

Good vs. bad:
  Smaller and shorter drawdowns indicate better capital preservation. The series is best analyzed together with drawdown duration and recovery metrics.

Caveats:
  Drawdown measures losses from previous peaks but does not show the path taken, volatility before the decline, or recovery speed.


### 11.2 `max_drawdown`

In [69]:
mdd = portfolio.metrics.max_drawdown(as_result=True)
print(repr(mdd))
mdd.explain()
note("max_drawdown")
note_explained("max_drawdown")

max_drawdown=-0.218643 % - moderate
max_drawdown (metric)

What it is:
  The largest observed loss from a historical peak to the following lowest point before a recovery or the end of the sample.

Formula:
  minimum(drawdown_series)

How to read it:
  Reported as a negative percentage. A value of -0.30 means the portfolio lost 30% from its previous peak at its worst point.

Good vs. bad:
  A smaller absolute drawdown is generally preferable because large losses require disproportionately larger gains to recover.

Caveats:
  Max drawdown only captures the worst event. It does not measure frequency, duration, or how quickly losses were recovered.

This result:
  moderate


### 11.3 `drawdown_duration`

In [70]:
duration = portfolio.metrics.drawdown_duration()
print(f"current drawdown_duration = {int(duration.iloc[-1])} periods since last peak")
print(f"longest drawdown_duration on record = {int(duration.max())} periods")
pexplain("drawdown_duration")
note("drawdown_duration")
note_explained("drawdown_duration")

current drawdown_duration = 304 periods since last peak
longest drawdown_duration on record = 304 periods
drawdown_duration (metric)

What it is:
  Tracks how long the portfolio has remained below its previous high after entering a drawdown period.

Formula:
  current_period - last_peak_period

How to read it:
  Higher values indicate longer recovery periods. The value resets when a new portfolio high is reached.

Good vs. bad:
  Shorter drawdown durations are generally preferable because investors recover losses faster.

Caveats:
  Duration does not measure the size of the loss. A portfolio can have a long shallow drawdown or a short severe drawdown.


### 11.4 `top_n_drawdowns`

In [71]:
top5 = portfolio.metrics.top_n_drawdowns(n=5)
print(top5)
pexplain("top_n_drawdowns")
note("top_n_drawdowns")
note_explained("top_n_drawdowns")

       start     trough        end   depth  duration_to_trough  duration_to_recovery  recovered
0 2025-10-06 2026-03-29        NaT -0.2186                 174                   NaN      False
1 2024-12-16 2025-04-08 2025-06-09 -0.2087                 113              175.0000       True
2 2024-07-16 2024-08-05 2024-09-24 -0.0990                  20               70.0000       True
3 2024-03-31 2024-05-01 2024-05-17 -0.0733                  31               47.0000       True
4 2023-08-08 2023-09-27 2023-11-01 -0.0727                  50               85.0000       True
top_n_drawdowns (metric)

What it is:
  A ranked table of the largest drawdown episodes, including their timing, depth, and recovery characteristics.

Formula:
  sort(drawdown_episodes, by=depth)

How to read it:
  The table identifies the worst historical loss periods and whether each drawdown eventually recovered.

Good vs. bad:
  Fewer severe drawdowns and faster recoveries generally indicate stronger downside resilie

### 11.5 `time_to_recovery`

In [72]:
ttr = portfolio.metrics.time_to_recovery()
print(f"time_to_recovery (worst episode) = {ttr!r} periods" + (" (not yet recovered)" if ttr is None else ""))
pexplain("time_to_recovery")
note("time_to_recovery")
note_explained("time_to_recovery")

time_to_recovery (worst episode) = None periods (not yet recovered)
time_to_recovery (metric)

What it is:
  Measures the number of periods required for the portfolio to recover from its largest drawdown back to its previous peak.

Formula:
  recovery_date - drawdown_start_date

How to read it:
  The value represents recovery time in the same frequency as the input data. None means the portfolio has not recovered by the end of the sample.

Good vs. bad:
  Shorter recovery periods indicate faster capital recovery. Long unrecovered periods increase investor risk and behavioral pressure.

Caveats:
  Recovery time depends on the observation period. A recent unresolved drawdown may appear worse simply because insufficient recovery time has passed.


### 11.6 `average_drawdown`

In [73]:
avg_dd = portfolio.metrics.average_drawdown(as_result=True)
print(repr(avg_dd))
avg_dd.explain()
note("average_drawdown")
note_explained("average_drawdown")

average_drawdown=-0.0240109 % - -2.40% average drawdown depth
average_drawdown (metric)

What it is:
  The average depth of completed drawdown episodes, showing the typical size of portfolio declines.

Formula:
  mean(drawdown_episode_depths)

How to read it:
  A value of -0.05 means the average drawdown episode reduced portfolio value by approximately 5%.

Good vs. bad:
  Values closer to zero indicate smaller typical losses. Compare with max_drawdown to identify whether risk is dominated by rare extreme events.

Caveats:
  Average drawdown ignores how long each episode lasts. A portfolio can have small but very persistent drawdowns.

This result:
  -2.40% average drawdown depth


### 11.7 `drawdown_at_risk`

In [74]:
for confidence in (0.95, 0.99):
    dar = portfolio.metrics.drawdown_at_risk(confidence=confidence)
    print(f"drawdown_at_risk[{confidence:.0%}] = {dar:.2%}")
portfolio.metrics.drawdown_at_risk(as_result=True).explain()
note("drawdown_at_risk")
note_explained("drawdown_at_risk")

drawdown_at_risk[95%] = -17.36%
drawdown_at_risk[99%] = -20.46%
drawdown_at_risk (metric)

What it is:
  A downside risk measure applying a percentile calculation to historical drawdown levels instead of returns.

Formula:
  percentile(drawdown_series, 100 * (1 - confidence))

How to read it:
  A value of -0.20 at 95% confidence means drawdowns exceeded 20% only during the worst 5% of observed periods.

Good vs. bad:
  Values closer to zero indicate lower historical drawdown severity.

Caveats:
  Like other percentile-based risk measures, this depends on the historical sample and may underestimate future extreme events.

This result:
  -17.36% drawdown threshold


## 12. Metrics -- Rolling & Expanding Windows (`portpy.metrics.rolling`)

### 12.1 `rolling_metric` (generic engine -- rolling any metric function)

In [75]:
from portpy.metrics.rolling import expanding_metric, rolling_metric

WINDOW = 60
roll_skew = rolling_metric(port_r, skewness, window=WINDOW)
print(f"rolling_metric(skewness, window={WINDOW}): {roll_skew.notna().sum()} valid values, "
      f"last={roll_skew.dropna().iloc[-1]:+.2f}")
pexplain("rolling_metric")
note("rolling_metric")
note_explained("rolling_metric")

rolling_metric(skewness, window=60): 1036 valid values, last=-0.06
rolling_metric (function)

What it is:
  Generic engine that applies any metric function repeatedly over a moving historical window.

Formula:
  metric(window_returns_t)

How to read it:
  Each output value represents the metric calculated using only the observations inside the current rolling window.

Good vs. bad:
  Useful for detecting changes in strategy behavior, risk, and performance characteristics over time.

Caveats:
  The usefulness depends on the metric being applied and the chosen window size. Short windows increase noise while long windows reduce responsiveness.


### 12.2 `rolling_sharpe`

In [76]:
roll_sharpe = portfolio.metrics.rolling_sharpe(window=WINDOW)
print(f"rolling_sharpe(window={WINDOW}) last 3:\n{roll_sharpe.dropna().tail(3)}")
pexplain("rolling_sharpe")
note("rolling_sharpe")
note_explained("rolling_sharpe")

rolling_sharpe(window=60) last 3:
Date
2026-08-04   2.1575
2026-08-05   2.3337
2026-08-06   1.9840
Name: rolling_sharpe, dtype: float64
rolling_sharpe (chart)

What it is:
  Sharpe ratio recalculated over a moving historical window to show whether risk-adjusted performance is stable, improving, or deteriorating through time.

Formula:
  (average_return - risk_free_rate) / standard_deviation_of_returns

How to read it:
  A consistently positive value indicates returns have compensated for risk during that period. Declining values indicate weakening risk-adjusted performance.

Good vs. bad:
  Prefer strategies with stable positive rolling Sharpe values rather than short periods of unusually high performance.

Caveats:
  Rolling Sharpe is sensitive to the chosen window size. Short windows react quickly but are noisy, while long windows are more stable but slower to detect changes.


### 12.3 `rolling_volatility`

In [77]:
roll_vol = portfolio.metrics.rolling_volatility(window=WINDOW)
print(f"rolling_volatility(window={WINDOW}) last={roll_vol.dropna().iloc[-1]:.2%}")
pexplain("rolling_volatility")
note("rolling_volatility")
note_explained("rolling_volatility")

rolling_volatility(window=60) last=17.28%
rolling_volatility (chart)

What it is:
  Volatility recalculated over a moving historical window to identify changes in risk levels, volatility clustering, and market regimes.

Formula:
  rolling_standard_deviation(returns) * sqrt(periods_per_year)

How to read it:
  Higher values indicate periods where returns are fluctuating more. Sudden increases often correspond to market stress or uncertainty.

Good vs. bad:
  Stable volatility is usually easier to manage. Rapid increases suggest rising portfolio risk and possible regime changes.

Caveats:
  Volatility measures past variability, not future risk. The selected rolling window strongly affects how quickly changes appear.


### 12.4 `rolling_beta`

In [78]:
roll_beta = portfolio.metrics.rolling_beta(benchmark=benchmark_returns, window=WINDOW)
print(f"rolling_beta(window={WINDOW}) last 3:\n{roll_beta.dropna().tail(3)}")
pexplain("rolling_beta")
note("rolling_beta")
note_explained("rolling_beta")

rolling_beta(window=60) last 3:
Date
2026-08-03   0.5726
2026-08-04   0.5739
2026-08-05   0.5757
Name: rolling_beta, dtype: float64
rolling_beta (chart)

What it is:
  Beta recalculated over a moving historical window to show how portfolio sensitivity to a benchmark changes over time.

Formula:
  covariance(asset_returns, benchmark_returns) / variance(benchmark_returns)

How to read it:
  Values above 1 indicate the portfolio tends to amplify benchmark movements. Values below 1 indicate lower sensitivity.

Good vs. bad:
  Stable beta values make risk management easier. Large shifts indicate changing market exposure or unstable relationships.

Caveats:
  Beta depends on the selected benchmark and rolling window. Historical relationships may break during market regime changes.


### 12.5 `rolling_correlation`

In [79]:
from portpy.metrics.rolling import rolling_correlation

roll_corr = rolling_correlation(master["BTC"].pct_change().dropna(), master["AAPL"].pct_change().dropna(), window=WINDOW)
print(f"rolling_correlation(BTC vs AAPL, window={WINDOW}) last={roll_corr.dropna().iloc[-1]:+.2f}")
pexplain("rolling_correlation")
note("rolling_correlation")
note_explained("rolling_correlation")

rolling_correlation(BTC vs AAPL, window=60) last=+0.26
rolling_correlation (chart)

What it is:
  Correlation between two return series recalculated over a moving historical window to show whether their relationship changes through time.

Formula:
  covariance(asset_a, asset_b) / (volatility_a * volatility_b)

How to read it:
  Values near 1 indicate the assets move together. Values near -1 indicate opposite movement. Values near 0 indicate weak relationship.

Good vs. bad:
  Lower correlation between portfolio assets generally improves diversification. Rising correlation can reduce diversification benefits.

Caveats:
  Correlation is dynamic and can increase sharply during market stress, reducing the protection expected from diversification.


### 12.6 `expanding_metric`

In [80]:
from portpy.metrics.performance import sharpe_ratio as _sharpe_ratio

exp_sharpe = expanding_metric(port_r, _sharpe_ratio, min_periods=30)
print(f"expanding_metric(sharpe_ratio): {exp_sharpe.notna().sum()} valid values, "
      f"first={exp_sharpe.dropna().iloc[0]:+.2f}, last={exp_sharpe.dropna().iloc[-1]:+.2f}")
pexplain("expanding_metric")
note("expanding_metric")
note_explained("expanding_metric")

expanding_metric(sharpe_ratio): 1066 valid values, first=-2.20, last=+1.00
expanding_metric (function)

What it is:
  Generic engine that applies any metric function using an expanding dataset that starts small and grows as new observations arrive.

Formula:
  metric(all_returns_available_until_t)

How to read it:
  The metric starts with limited information and becomes more stable as more observations accumulate.

Good vs. bad:
  Useful for tracking how estimates converge over time and how early performance compares with long-term history.

Caveats:
  Early values can be unreliable because they are calculated from very few observations.


## 13. Metrics -- Distributions (`portpy.metrics.distributions`)

### 13.1 `describe`

In [81]:
desc = portfolio.metrics.describe()
print(desc)
pexplain("describe")
note("describe")
note_explained("describe")

count      1,095.0000
mean           0.0006
std            0.0095
min           -0.0456
25%           -0.0039
50%            0.0006
75%            0.0052
max            0.0776
skew           0.3591
kurtosis       6.1098
dtype: float64
describe (metric)

What it is:
  A descriptive summary of the return distribution, including central tendency, dispersion, and tail behavior.

Formula:
  pandas.Series.describe() + skewness/kurtosis

How to read it:
  Use this as a quick sanity check for the return sample: typical values, spread, and extreme observations.

Good vs. bad:
  A well-behaved distribution should show reasonable center and spread, but there is no universal 'good' threshold for descriptive stats alone.


### 13.2 `normality_test`

In [82]:
nt = portfolio.metrics.normality_test()
print(nt)

# normality_test() stamps a "_portpy_explain_name" *dict key* (not an attribute), but
# explain()'s dispatch checks `hasattr(obj, "_portpy_explain_name")`, which a plain dict
# never satisfies -- so dispatching on the dict itself raises, and the registered name
# has to be passed explicitly instead. Demonstrated here as a known quirk, not hidden.
try:
    pexplain(nt)
except TypeError as exc:
    print(f"\n(known quirk) explain(dict_with_marker_key) raises: {exc}")
pexplain("normality_test", value=nt)
note("normality_test")
note_explained("normality_test")

{'statistic': 1708.0955397722555, 'p_value': 0.0, 'is_normal': False, '_portpy_explain_name': 'normality_test'}

(known quirk) explain(dict_with_marker_key) raises: Don't know how to explain an object of type 'dict'. Pass a registered name (str), a MetricResult, or a PortPy chart/model/strategy.
normality_test (metric)

What it is:
  Jarque-Bera test for whether the return distribution's skewness and kurtosis match a Normal distribution.

Formula:
  statistic based on sample skewness and excess kurtosis; p-value from a chi-squared(2) reference distribution

How to read it:
  A small p-value (conventionally < 0.05) means the data significantly deviates from Normal - which is the norm, not the exception, for real returns.

Good vs. bad:
  Neither outcome is 'good' or 'bad' on its own - it's a diagnostic. Rejecting normality is a signal to prefer historical/Cornish-Fisher VaR over parametric (Normal-assumption) VaR.

This result:
  p=0.0000 -> reject Normality (fat tails/skew likely - use

### 13.3 `best_worst_periods`

In [83]:
bw = portfolio.metrics.best_worst_periods(n=5)
print("best:\n", bw["best"])
print("\nworst:\n", bw["worst"])
pexplain("best_worst_periods")
note("best_worst_periods")
note_explained("best_worst_periods")

best:
 Date
2025-04-09   0.0776
2026-02-06   0.0445
2024-08-08   0.0408
2024-03-20   0.0324
2025-05-08   0.0321
Name: All-Weather Multi-Asset Portfolio, dtype: float64

worst:
 Date
2026-02-05   -0.0456
2024-08-05   -0.0389
2024-12-18   -0.0340
2026-01-29   -0.0338
2025-10-10   -0.0332
Name: All-Weather Multi-Asset Portfolio, dtype: float64
best_worst_periods (metric)

What it is:
  The single best and worst individual return observations in the sample.

How to read it:
  Useful for a gut check: do the extremes correspond to known market events, or do they look like data errors?

Good vs. bad:
  Not a graded metric - a diagnostic/sanity-check tool.


### 13.4 `win_rate`

In [84]:
wr = portfolio.metrics.win_rate(as_result=True)
print(repr(wr))
wr.explain()
note("win_rate")
note_explained("win_rate")

win_rate=0.551598 % - 55.2% of periods were positive
win_rate (metric)

What it is:
  The fraction of periods (days/weeks/months) with a positive return.

Formula:
  count(r > 0) / count(r)

How to read it:
  0.55 means 55% of periods were profitable.

Good vs. bad:
  Above 50% is intuitively 'good', but a high win rate with small wins and rare huge losses can still be a bad strategy overall (see win_loss_ratio) - never judge win rate alone.

This result:
  55.2% of periods were positive


### 13.5 `win_loss_ratio`

In [85]:
wlr = portfolio.metrics.win_loss_ratio()
print(f"win_loss_ratio = {wlr:.3f}")
portfolio.metrics.win_loss_ratio(as_result=True).explain()
note("win_loss_ratio")
note_explained("win_loss_ratio")

win_loss_ratio = 0.977
win_loss_ratio (metric)

What it is:
  How big the average win is relative to the average loss, in magnitude.

Formula:
  mean(r for r > 0) / |mean(r for r < 0)|

How to read it:
  A ratio of 2.0 means winning periods are, on average, twice as large as losing periods.

Good vs. bad:
  Read together with win_rate: a strategy can be profitable with a win rate below 50% if this ratio is high enough (small frequent losses, larger rare wins), and vice versa.

This result:
  0.98 (average win smaller than average loss - relies on high win rate)


### 13.6 `positive_periods_pct` (alias of `win_rate`)

In [86]:
ppp = portfolio.metrics.positive_periods_pct()
print(f"positive_periods_pct = {ppp:.2%}  (matches win_rate: {np.isclose(ppp, float(wr))})")
portfolio.metrics.positive_periods_pct(as_result=True).explain()
note("positive_periods_pct")
note_explained("positive_periods_pct")

positive_periods_pct = 55.16%  (matches win_rate: True)
positive_periods_pct (metric)

What it is:
  Same as win_rate: the fraction of periods with a positive return.

How to read it:
  See win_rate.

Good vs. bad:
  See win_rate.

This result:
  55.2% of periods were positive


### 13.7 `monthly_returns_table`

In [87]:
monthly_table = portfolio.metrics.monthly_returns_table()
print(monthly_table.tail(6))
pexplain("monthly_returns_table")
note("monthly_returns_table")
note_explained("monthly_returns_table")

         Jan     Feb     Mar     Apr    May     Jun    Jul     Aug     Sep     Oct     Nov     Dec    Year
year                                                                                                      
2023     NaN     NaN     NaN     NaN    NaN     NaN    NaN -0.0245 -0.0369  0.0557  0.1117  0.0574  0.1658
2024  0.0081  0.1153  0.0541 -0.0691 0.0872  0.0171 0.0261 -0.0119  0.0302  0.0044  0.1287 -0.0235  0.4107
2025  0.0348 -0.0554 -0.0425  0.0342 0.0789  0.0310 0.0634  0.0281  0.0403 -0.0014 -0.0480 -0.0082  0.1546
2026 -0.0542 -0.0314 -0.0376  0.0648 0.0183 -0.0973 0.0929  0.0259     NaN     NaN     NaN     NaN -0.0324
monthly_returns_table (chart)

What it is:
  Compounded return for every calendar month, laid out as a year x month grid (the data behind a monthly-returns heatmap).

How to read it:
  Each cell is that month's compounded return; the 'Year' column is the full calendar year's compounded return. Read row-by-row to spot seasonal patterns or bad years; column-

### 13.8 `return_histogram_data`

In [88]:
counts, edges = portfolio.metrics.return_histogram_data(bins=25)
print(f"return_histogram_data: {len(counts)} bins, busiest bin count={counts.max()}, "
      f"range=[{edges[0]:+.3%}, {edges[-1]:+.3%}]")
pexplain("return_histogram_data")
note("return_histogram_data")
note_explained("return_histogram_data")

return_histogram_data: 25 bins, busiest bin count=367, range=[-4.564%, +7.760%]
return_histogram_data (chart)

What it is:
  The raw bin counts and edges describing the shape of the return distribution.

How to read it:
  A tall, narrow, symmetric hump centered near (or slightly above) zero is 'textbook'. Look for a long left tail (crash risk) or a bimodal shape (regime-switching behavior).

Good vs. bad:
  Not graded directly - use skewness/kurtosis/normality_test for quantitative judgments about the shape shown here.


## 14. Metrics -- Benchmark-Relative (`portpy.metrics.benchmarks`)

### 14.1 `alpha`

In [89]:
alpha_result = portfolio.metrics.alpha(benchmark=benchmark_returns, as_result=True)
print(repr(alpha_result))
alpha_result.explain()
note("alpha")
note_explained("alpha")

alpha=0.0277204 % - +2.8%/yr (positive excess performance)
alpha (metric)

What it is:
  The annualized excess return generated by the portfolio after accounting for the return expected from its benchmark exposure through beta.

Formula:
  (1 + mean(excess_return) - beta * mean(benchmark_excess_return)) ** periods_per_year - 1

How to read it:
  Positive alpha means the portfolio produced returns beyond what its benchmark exposure would explain. Negative alpha means underperformance after adjusting for market exposure.

Good vs. bad:
  Positive and persistent alpha is desirable because it indicates value added beyond benchmark exposure.

Caveats:
  Alpha depends heavily on the chosen benchmark and beta estimate. A poor benchmark can make unrelated returns appear as alpha.

This result:
  +2.8%/yr (positive excess performance)


### 14.2 `correlation`

In [90]:
corr_vs_spy = portfolio.metrics.correlation(benchmark=benchmark_returns)
print(f"correlation vs SPY = {corr_vs_spy:+.3f}")
portfolio.metrics.correlation(benchmark=benchmark_returns, as_result=True).explain()
note("correlation")
note_explained("correlation")

correlation vs SPY = +0.712
correlation (metric)

What it is:
  Measures the linear relationship between portfolio returns and benchmark returns, showing how closely they move together.

Formula:
  covariance(returns, benchmark) / (std(returns) * std(benchmark))

How to read it:
  Values close to 1 indicate the portfolio usually moves with the benchmark. Values close to -1 indicate opposite movement. Values near 0 indicate weak linear relationship.

Good vs. bad:
  High correlation is desirable for benchmark-tracking strategies. Lower correlation is preferable when seeking diversification from the benchmark.

Caveats:
  Correlation only measures linear relationships and can change significantly during different market environments.

This result:
  +0.71 (strong relationship)


### 14.3 `r_squared`

In [91]:
r2 = portfolio.metrics.r_squared(benchmark=benchmark_returns)
print(f"r_squared vs SPY = {r2:.2%}  (matches correlation^2: {np.isclose(r2, corr_vs_spy ** 2)})")
portfolio.metrics.r_squared(benchmark=benchmark_returns, as_result=True).explain()
note("r_squared")
note_explained("r_squared")

r_squared vs SPY = 50.70%  (matches correlation^2: True)
r_squared (metric)

What it is:
  Measures how much of the portfolio return variability can be statistically explained by benchmark movements.

Formula:
  correlation(returns, benchmark) ** 2

How to read it:
  A value of 0.80 means 80% of return variation is associated with benchmark movements, while 20% comes from other sources.

Good vs. bad:
  Higher values indicate stronger benchmark dependence. This is useful for index tracking but may be undesirable for actively differentiated strategies.

Caveats:
  R-squared does not measure whether returns are positive or negative. A portfolio can have high R-squared and still perform poorly.

This result:
  51% of variance explained by benchmark


### 14.4 `up_capture_ratio`

In [92]:
up_cap = portfolio.metrics.up_capture_ratio(benchmark=benchmark_returns)
print(f"up_capture_ratio = {up_cap:.2f}")
portfolio.metrics.up_capture_ratio(benchmark=benchmark_returns, as_result=True).explain()
note("up_capture_ratio")
note_explained("up_capture_ratio")

up_capture_ratio = 0.70
up_capture_ratio (metric)

What it is:
  Measures how much of the benchmark's positive performance the portfolio captures during periods when the benchmark rises.

Formula:
  annualized_return(portfolio_returns_when_benchmark_positive) / annualized_return(benchmark_returns_when_positive)

How to read it:
  A value above 1 means the portfolio gains more than the benchmark during positive benchmark periods.

Good vs. bad:
  Above 1 is generally desirable because it indicates stronger participation in market gains.

Caveats:
  Requires enough positive benchmark periods. Results can be distorted by a small number of strong market moves.

This result:
  70% of benchmark upside captured


### 14.5 `down_capture_ratio`

In [93]:
down_cap = portfolio.metrics.down_capture_ratio(benchmark=benchmark_returns)
print(f"down_capture_ratio = {down_cap:.2f}")
portfolio.metrics.down_capture_ratio(benchmark=benchmark_returns, as_result=True).explain()
note("down_capture_ratio")
note_explained("down_capture_ratio")

down_capture_ratio = 0.96
down_capture_ratio (metric)

What it is:
  Measures how much of the benchmark's negative performance the portfolio experiences during periods when the benchmark falls.

Formula:
  annualized_return(portfolio_returns_when_benchmark_negative) / annualized_return(benchmark_returns_when_negative)

How to read it:
  A value below 1 means the portfolio loses less than the benchmark during declining periods.

Good vs. bad:
  Below 1 is generally desirable because it indicates downside protection.

Caveats:
  The ratio can behave unexpectedly when benchmark losses are small or when the sample contains few negative periods.

This result:
  96% of benchmark downside captured (defensive)


### 14.6 `capture_ratio`

In [94]:
overall_cap = portfolio.metrics.capture_ratio(benchmark=benchmark_returns)
print(f"capture_ratio (unconditional) = {overall_cap:.2f}")
portfolio.metrics.capture_ratio(benchmark=benchmark_returns, as_result=True).explain()
note("capture_ratio")
note_explained("capture_ratio")

capture_ratio (unconditional) = 0.89
capture_ratio (metric)

What it is:
  Compares the portfolio's overall annualized return against the benchmark's annualized return without separating positive and negative periods.

Formula:
  annualized_return(portfolio) / annualized_return(benchmark)

How to read it:
  A value above 1 means the portfolio produced more annualized return than the benchmark.

Good vs. bad:
  Higher values indicate stronger relative performance, but they should be interpreted together with up and down capture ratios.

Caveats:
  A single ratio hides the path taken to achieve returns. Two portfolios can have the same capture ratio with very different risk profiles.

This result:
  0.89x benchmark return


### 14.7 `batting_average`

In [95]:
ba = portfolio.metrics.batting_average(benchmark=benchmark_returns)
print(f"batting_average vs SPY = {ba:.1%}")
portfolio.metrics.batting_average(benchmark=benchmark_returns, as_result=True).explain()
note("batting_average")
note_explained("batting_average")

batting_average vs SPY = 48.3%
batting_average (metric)

What it is:
  Measures the percentage of periods where the portfolio outperformed the benchmark.

Formula:
  count(portfolio_return > benchmark_return) / count(periods)

How to read it:
  A value of 0.55 means the portfolio beat the benchmark in 55% of observed periods.

Good vs. bad:
  A higher batting average indicates more frequent relative wins, but it does not measure the size of those wins or losses.

Caveats:
  A portfolio can have a low batting average and still outperform if a small number of gains are large enough. Combine with return-based metrics.

This result:
  outperformed benchmark in 48% of periods


## 15. Metrics -- Regressions (`portpy.metrics.regressions`)

### 15.1 `linear_regression` -- single-factor (vs. SPY)

In [96]:
model_1f = portfolio.metrics.linear_regression(x=benchmark_returns)
print(model_1f.summary())
pexplain("linear_regression")
note("linear_regression")
note_explained("linear_regression")

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.507
Model:                            OLS   Adj. R-squared:                  0.506
Method:                 Least Squares   F-statistic:                     770.2
Date:                Wed, 05 Aug 2026   Prob (F-statistic):          3.92e-117
Time:                        23:07:36   Log-Likelihood:                 2587.7
No. Observations:                 751   AIC:                            -5171.
Df Residuals:                     749   BIC:                            -5162.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       9.424e-05      0.000      0.333      0.7

### 15.2 `linear_regression` -- multi-factor (SPY + GLD)

In [97]:
gld_returns = master["GLD"].pct_change().dropna().rename("GLD")
factors = pd.concat([benchmark_returns.rename("SPY"), gld_returns], axis=1)
model_2f = portfolio.metrics.linear_regression(x=factors)
print(f"Multi-factor R^2={model_2f.rsquared:.3f}, coefficients:\n{model_2f.params}")

Multi-factor R^2=0.526, coefficients:
const   -0.0000
SPY      0.7797
GLD      0.1169
dtype: float64


### 15.3 `regression_summary`

In [98]:
from portpy.metrics.regressions import regression_summary

summary_1f = regression_summary(model_1f)
summary_2f = regression_summary(model_2f)
print("Single-factor summary:\n", summary_1f)
print(f"R^2={summary_1f.attrs['r_squared']:.3f}, adj R^2={summary_1f.attrs['adj_r_squared']:.3f}\n")
print("Two-factor summary:\n", summary_2f)
pexplain("regression_summary")
note("regression_summary")
note_explained("regression_summary")

Single-factor summary:
         coef  std_err  t_stat  p_value  conf_low  conf_high
const 0.0001   0.0003  0.3331   0.7391   -0.0005     0.0006
SPY   0.8072   0.0291 27.7519   0.0000    0.7501     0.8643
R^2=0.507, adj R^2=0.506

Two-factor summary:
          coef  std_err  t_stat  p_value  conf_low  conf_high
const -0.0000   0.0003 -0.0507   0.9596   -0.0006     0.0005
SPY    0.7797   0.0290 26.9053   0.0000    0.7228     0.8366
GLD    0.1169   0.0214  5.4735   0.0000    0.0750     0.1588
regression_summary (function)

What it is:
  A structured summary table of an OLS regression model containing coefficient estimates, uncertainty measures, statistical tests, and confidence intervals.

Formula:
  t_stat = coef / std_err; confidence_interval = coef +/- critical_value * std_err

How to read it:
  Each row represents an explanatory variable. coef shows the estimated relationship between the predictor and target variable while controlling for other variables. std_err measures uncertainty 

### 15.4 `rolling_regression`

In [99]:
rolling_coefs = portfolio.metrics.rolling_regression(x=benchmark_returns, window=WINDOW)
print(f"rolling_regression(window={WINDOW}): {rolling_coefs.shape}")
print(rolling_coefs.tail(3))
pexplain("rolling_regression")
note("rolling_regression")
note_explained("rolling_regression")

rolling_regression(window=60): (692, 2)
             const    SPY
Date                     
2026-08-03 -0.0014 0.5726
2026-08-04 -0.0013 0.5739
2026-08-05 -0.0012 0.5757
rolling_regression (function)

What it is:
  Refits an OLS regression over a rolling window so you can inspect how coefficient estimates evolve through time.

Formula:
  same as linear_regression, but estimated on a trailing window

How to read it:
  Use this to see whether the relationship between variables changes over time, which is common in financial data.

Good vs. bad:
  Stable coefficients over time are generally more reliable than coefficients that drift dramatically.


## 16. Metrics -- Covariance & Risk Decomposition (`portpy.metrics.covariance`)

### 16.1 `covariance_matrix` (raw vs. annualized)

In [100]:
from portpy.metrics.covariance import (
    component_contribution_to_risk, correlation_matrix, covariance_matrix,
    diversification_ratio, marginal_contribution_to_risk, portfolio_variance, portfolio_volatility,
)

cov_raw = portfolio.metrics.covariance_matrix(annualized=False)
cov_ann = portfolio.metrics.covariance_matrix(annualized=True)
print("covariance_matrix(annualized=True):\n", cov_ann)
assert np.allclose(cov_ann.to_numpy(), cov_raw.to_numpy() * portfolio.frequency)
pexplain("covariance_matrix")
note("covariance_matrix")
note_explained("covariance_matrix")

covariance_matrix(annualized=True):
        AAPL     GLD     JPM    MSFT     TLT    VNQ     BTC    ETH     SAP
AAPL 0.0719  0.0036  0.0174  0.0247  0.0040 0.0150  0.0186 0.0318  0.0090
GLD  0.0036  0.0435  0.0034  0.0033  0.0037 0.0057  0.0109 0.0137 -0.0024
JPM  0.0174  0.0034  0.0530  0.0140 -0.0012 0.0144  0.0253 0.0354  0.0037
MSFT 0.0247  0.0033  0.0140  0.0677 -0.0004 0.0063  0.0275 0.0406  0.0284
TLT  0.0040  0.0037 -0.0012 -0.0004  0.0184 0.0099 -0.0009 0.0004  0.0007
VNQ  0.0150  0.0057  0.0144  0.0063  0.0099 0.0283  0.0138 0.0201  0.0047
BTC  0.0186  0.0109  0.0253  0.0275 -0.0009 0.0138  0.2203 0.2508  0.0086
ETH  0.0318  0.0137  0.0354  0.0406  0.0004 0.0201  0.2508 0.4340  0.0133
SAP  0.0090 -0.0024  0.0037  0.0284  0.0007 0.0047  0.0086 0.0133  0.1022
covariance_matrix (metric)

What it is:
  A matrix showing how asset returns move together, including both individual asset volatility and relationships between assets.

Formula:
  covariance(returns_i, returns_j)

How to r

### 16.2 `correlation_matrix`

In [101]:
corr_matrix = portfolio.metrics.correlation_matrix()
print(corr_matrix.round(2))
pexplain("correlation_matrix")
note("correlation_matrix")
note_explained("correlation_matrix")

       AAPL     GLD     JPM    MSFT     TLT    VNQ     BTC    ETH     SAP
AAPL 1.0000  0.0600  0.2800  0.3500  0.1100 0.3300  0.1500 0.1800  0.1100
GLD  0.0600  1.0000  0.0700  0.0600  0.1300 0.1600  0.1100 0.1000 -0.0400
JPM  0.2800  0.0700  1.0000  0.2300 -0.0400 0.3700  0.2300 0.2300  0.0500
MSFT 0.3500  0.0600  0.2300  1.0000 -0.0100 0.1500  0.2300 0.2400  0.3400
TLT  0.1100  0.1300 -0.0400 -0.0100  1.0000 0.4300 -0.0100 0.0100  0.0200
VNQ  0.3300  0.1600  0.3700  0.1500  0.4300 1.0000  0.1800 0.1800  0.0900
BTC  0.1500  0.1100  0.2300  0.2300 -0.0100 0.1800  1.0000 0.8100  0.0600
ETH  0.1800  0.1000  0.2300  0.2400  0.0100 0.1800  0.8100 1.0000  0.0600
SAP  0.1100 -0.0400  0.0500  0.3400  0.0200 0.0900  0.0600 0.0600  1.0000
correlation_matrix (metric)

What it is:
  A matrix showing the strength and direction of relationships between asset returns, independent of their individual volatility levels.

Formula:
  correlation(returns_i, returns_j)

How to read it:
  Values range from

### 16.3 `portfolio_variance`

In [102]:
pvar = portfolio.metrics.portfolio_variance(as_result=True)
print(repr(pvar))
pvar.explain()
note("portfolio_variance")
note_explained("portfolio_variance")

portfolio_variance=9.02729e-05
portfolio_variance (metric)

What it is:
  The total portfolio return variance, measuring how much portfolio returns fluctuate based on individual asset variances and how assets move together.

Formula:
  w @ Cov @ w

How to read it:
  Variance is expressed in squared return units. It is mainly useful as an intermediate calculation because volatility is easier to interpret.

Good vs. bad:
  Lower variance means lower absolute portfolio risk, but it should always be evaluated together with expected return and investment objectives.

Caveats:
  Variance is sensitive to the covariance estimate. Short datasets or unstable correlations can produce unreliable risk estimates.


### 16.4 `portfolio_volatility`

In [103]:
pvol = portfolio.metrics.portfolio_volatility(as_result=True)
print(repr(pvol))
assert np.isclose(float(pvol), float(volatility(port_r, annualized=False)), atol=1e-3)
pvol.explain()
note("portfolio_volatility")
note_explained("portfolio_volatility")

portfolio_volatility=0.0095012 - 0.95%
portfolio_volatility (metric)

What it is:
  The portfolio's total return volatility, representing the expected dispersion of portfolio returns around their average return.

Formula:
  sqrt(w @ Cov @ w)

How to read it:
  Expressed as a percentage, volatility can be directly compared with the volatility of individual assets or benchmarks.

Good vs. bad:
  Lower volatility generally means lower risk, but higher volatility can be acceptable when compensated by higher expected returns.

Caveats:
  Historical volatility does not predict future volatility. Market regimes, correlations, and liquidity conditions can change.

This result:
  0.95%


### 16.5 `diversification_ratio`

In [104]:
div_ratio = portfolio.metrics.diversification_ratio(as_result=True)
print(repr(div_ratio))

# Also works with an explicit, overridden cov_matrix (e.g. an annualized one).
div_ratio_annualized_cov = portfolio.metrics.diversification_ratio(cov_matrix=cov_ann)
print(f"diversification_ratio with an explicit annualized cov_matrix override = {div_ratio_annualized_cov:.3f}")
div_ratio.explain()
note("diversification_ratio")
note_explained("diversification_ratio")

diversification_ratio=1.69095 - 1.69x diversification benefit
diversification_ratio with an explicit annualized cov_matrix override = 1.691
diversification_ratio (metric)

What it is:
  Measures how much diversification benefit a portfolio receives from combining assets with different volatility and correlation characteristics.

Formula:
  (sum(weight_i * asset_volatility_i)) / portfolio_volatility

How to read it:
  A value above 1 means the combined portfolio is less risky than the weighted average standalone asset risks.

Good vs. bad:
  Higher values indicate stronger diversification benefits. A value close to 1 indicates little or no diversification advantage.

Caveats:
  The ratio depends on the quality of the covariance matrix. Poor correlation estimates can overstate diversification benefits.

This result:
  1.69x diversification benefit


### 16.6 `marginal_contribution_to_risk`

In [105]:
mctr = portfolio.metrics.marginal_contribution_to_risk()
print(pd.Series(mctr, index=portfolio.asset_names, name="MCTR"))
pexplain("marginal_contribution_to_risk")
note("marginal_contribution_to_risk")
note_explained("marginal_contribution_to_risk")

AAPL   0.0071
GLD    0.0027
JPM    0.0055
MSFT   0.0072
TLT    0.0010
VNQ    0.0038
BTC    0.0202
ETH    0.0285
SAP    0.0045
Name: MCTR, dtype: float64
marginal_contribution_to_risk (metric)

What it is:
  Measures how much portfolio volatility changes when the allocation to one asset changes slightly.

Formula:
  (Cov @ w) / portfolio_volatility

How to read it:
  Each value represents the incremental volatility impact of increasing one asset's portfolio weight.

Good vs. bad:
  Assets with lower or negative marginal risk contribution can help reduce portfolio risk. Large positive values indicate assets driving portfolio volatility.

Caveats:
  Marginal risk contribution depends on current portfolio weights and correlations. It is not the same as standalone asset volatility.


### 16.7 `component_contribution_to_risk`

In [106]:
cctr = portfolio.metrics.component_contribution_to_risk()
breakdown = pd.DataFrame({
    "weight": portfolio.weights,
    "MCTR": mctr,
    "CCTR": cctr,
    "CCTR_pct_of_total": cctr / cctr.sum(),
})
print(breakdown.round(4))
print(f"\nsum(CCTR) == portfolio_volatility: {np.isclose(cctr.sum(), float(pvol))}")
pexplain("component_contribution_to_risk")
note("component_contribution_to_risk")
note_explained("component_contribution_to_risk")

      weight   MCTR   CCTR  CCTR_pct_of_total
AAPL  0.1500 0.0071 0.0011             0.1120
GLD   0.1000 0.0027 0.0003             0.0286
JPM   0.1000 0.0055 0.0005             0.0576
MSFT  0.1300 0.0072 0.0009             0.0987
TLT   0.1000 0.0010 0.0001             0.0109
VNQ   0.1000 0.0038 0.0004             0.0404
BTC   0.1500 0.0202 0.0030             0.3186
ETH   0.1000 0.0285 0.0029             0.3003
SAP   0.0700 0.0045 0.0003             0.0329

sum(CCTR) == portfolio_volatility: True
component_contribution_to_risk (metric)

What it is:
  Breaks down total portfolio volatility into the amount of risk contributed by each individual asset position.

Formula:
  weight_i * marginal_contribution_to_risk_i

How to read it:
  Each component shows the portion of total portfolio volatility attributable to one asset. The components add up to total portfolio volatility.

Good vs. bad:
  A balanced risk contribution means no single asset dominates portfolio risk. Concentrated contributi

## 17. Metrics -- One-Shot Summaries (`portpy.metrics.summary`)

### 17.1 `tearsheet_summary`

In [107]:
tearsheet = portfolio.metrics.tearsheet_summary()
print("=" * 60)
print(f"{portfolio.name.upper()} -- TEARSHEET SUMMARY")
print("=" * 60)
for metric_name, result in tearsheet.items():
    print(f"  {metric_name:22s} {float(result):>10.4f}   ({result.interpretation})")
pexplain("tearsheet_summary")
note("tearsheet_summary")
note_explained("tearsheet_summary")

ALL-WEATHER MULTI-ASSET PORTFOLIO -- TEARSHEET SUMMARY
  total_return               0.8372   (+83.7% total return)
  annualized_return          0.2248   (+22.5% annualized return)
  cagr                       0.2248   (+22.5%/year compounded)
  volatility                 0.1815   (18.2%/yr (moderate))
  sharpe_ratio               1.0062   (good)
  sortino_ratio              1.8063   (good)
  calmar_ratio               1.0280   (solid)
  max_drawdown              -0.2186   (moderate)
  value_at_risk_95          -0.0150   (-1.50% - expect a worse-than-this loss only in the excluded tail probability)
  conditional_var_95        -0.0215   (-2.15% average loss in the worst-case tail)
  skewness                   0.3591   (+0.36 (roughly symmetric))
  kurtosis                   6.1098   (+6.11 (fat tails - Normal-based risk estimates will understate real risk))
  win_rate                   0.5516   (55.2% of periods were positive)
tearsheet_summary (metric)

What it is:
  A complete portfoli

### 17.2 `compare_to_benchmark`

In [108]:
comparison = portfolio.metrics.compare_to_benchmark(benchmark=benchmark_returns)
print(comparison)
pexplain("compare_to_benchmark")
note("compare_to_benchmark")
note_explained("compare_to_benchmark")

                    portfolio  benchmark  difference
annualized_return      0.2851     0.3209     -0.0358
volatility             0.2101     0.1853      0.0248
sharpe_ratio           1.1249     1.3970     -0.2722
max_drawdown          -0.2217    -0.1876     -0.0342
beta                   0.8072        NaN         NaN
alpha                  0.0277        NaN         NaN
correlation            0.7120        NaN         NaN
information_ratio     -0.1487        NaN         NaN
up_capture_ratio       0.7015        NaN         NaN
down_capture_ratio     0.9609        NaN         NaN
batting_average        0.4834        NaN         NaN
compare_to_benchmark (metric)

What it is:
  A structured comparison between a portfolio and a benchmark showing absolute performance, risk characteristics, and benchmark-relative statistics.

Formula:
  comparison = portfolio_metrics - benchmark_metrics + relative_metrics

How to read it:
  Positive differences generally indicate portfolio outperformance for re

## 18. Metrics -- Transaction Costs (`portpy.metrics.costs`)

`Portfolio` itself assumes frictionless, constant-weight rebalancing every period (see
`.returns()`'s docstring) -- `portpy.strategies` (a v0.1.0 stub, see section 21.2) would be
where real rebalancing/turnover simulation eventually lives. For now we simulate a simple
buy-and-hold-with-periodic-rebalancing weight history by hand, purely to have realistic
input for `turnover_from_weights`.

### 18.1 `turnover_from_weights`

In [109]:
from portpy.metrics.costs import net_of_costs_returns, turnover_from_weights


def simulate_weight_drift(returns_df: pd.DataFrame, target: pd.Series, rebalance_every: int = 21) -> pd.DataFrame:
    aligned = returns_df[target.index]
    history = []
    current = target.copy()
    for i, date in enumerate(aligned.index):
        if i > 0 and i % rebalance_every == 0:
            current = target.copy()
        history.append(current.copy())
        port_r = float((current * aligned.loc[date]).sum())
        current = current * (1.0 + aligned.loc[date]) / (1.0 + port_r)
    return pd.DataFrame(history, index=aligned.index)


weights_history = simulate_weight_drift(portfolio.asset_returns(), portfolio.weights, rebalance_every=21)
turnover = turnover_from_weights(weights_history)
print(f"turnover_from_weights: mean={turnover.mean():.2%}, max={turnover.max():.2%} "
      f"(spikes on rebalance days, drifts near 0 between them)")
print(turnover.tail(5))
pexplain("turnover_from_weights")
note("turnover_from_weights")
note_explained("turnover_from_weights")

turnover_from_weights: mean=0.62%, max=50.00% (spikes on rebalance days, drifts near 0 between them)
Date
2026-08-02   0.0008
2026-08-03   0.0028
2026-08-04   0.0262
2026-08-05   0.0022
2026-08-06   0.0039
Name: turnover, dtype: float64
turnover_from_weights (metric)

What it is:
  Measures how much of the portfolio is traded between periods by analyzing changes in portfolio allocation weights.

Formula:
  0.5 * sum(abs(current_weight - previous_weight))

How to read it:
  A turnover value of 0.20 means that 20% of portfolio value was exchanged during that period. The metric reflects portfolio trading activity rather than investment performance.

Good vs. bad:
  Lower turnover usually means lower implementation costs and less trading drag. Higher turnover can be justified when the strategy generates enough excess return to compensate for additional costs.

Caveats:
  Weight-based turnover does not estimate actual execution costs. It ignores market impact, spreads, commissions, taxes, l

### 18.2 `net_of_costs_returns`

In [110]:
net_variable = net_of_costs_returns(port_r, turnover, cost_bps=10)
net_constant = net_of_costs_returns(port_r, 0.05, cost_bps=10)

gross_total = total_return(port_r)
net_variable_total = total_return(net_variable)
net_constant_total = total_return(net_constant)
print(f"gross total_return                = {gross_total:+.2%}")
print(f"net (variable turnover, 10bps)     = {net_variable_total:+.2%}")
print(f"net (constant 5% turnover, 10bps)  = {net_constant_total:+.2%}")
pexplain("net_of_costs_returns")
note("net_of_costs_returns")
note_explained("net_of_costs_returns")

gross total_return                = +83.72%
net (variable turnover, 10bps)     = +82.49%
net (constant 5% turnover, 10bps)  = +73.94%
net_of_costs_returns (metric)

What it is:
  Adjusts portfolio returns by subtracting estimated transaction costs to show performance after implementation expenses.

Formula:
  gross_return - turnover * (cost_bps / 10000)

How to read it:
  Compare net returns with gross returns to understand how much performance is consumed by trading costs.

Good vs. bad:
  A small difference between gross and net returns indicates a strategy is more robust to implementation costs. A large difference suggests trading activity significantly reduces realized performance.

Caveats:
  The estimate assumes a simplified constant transaction cost. Real trading costs vary with liquidity, order size, market conditions, execution method, and asset characteristics.


## 19. The Explainability Layer (`portpy.explain`)

### 19.1 `available()` and `get()` -- the registry

In [111]:
print("Registered explanation names by category:")
for category in ("metric", "chart", "model", "strategy", "function"):
    names = available(category)
    print(f"  {category:10s}: {len(names)} registered")
print(f"\nTotal registered: {len(available())}")

card = get("max_drawdown")
print(f"\nget('max_drawdown') -> name={card.name}, category={card.category}")
print(card.summary)

try:
    get("not_a_real_metric")
except KeyError as exc:
    print(f"\nExpected KeyError (unregistered name): {exc}")

Registered explanation names by category:
  metric    : 67 registered
  chart     : 6 registered
  model     : 0 registered
  strategy  : 0 registered
  function  : 5 registered

Total registered: 78

get('max_drawdown') -> name=max_drawdown, category=metric
The largest observed loss from a historical peak to the following lowest point before a recovery or the end of the sample.

Expected KeyError (unregistered name): "No explanation registered for 'not_a_real_metric'. Call portpy.explain.available() to list all 78 registered names."


### 19.2 `Explanation` dataclass -- construct, `.render()`, and `register()` a custom one

In [112]:
custom_explanation = Explanation(
    name="debug_notebook_custom_metric",
    category="metric",
    summary="A placeholder custom metric registered from this notebook, to demonstrate the registry is extensible.",
    formula="n/a (demo only)",
    how_to_read="This exists purely to show `register()` accepts new Explanation objects at runtime.",
    good_vs_bad="n/a",
    caveats="Not a real metric -- don't use this outside this notebook.",
    interpret=lambda v: f"demo value = {v:.2f}",
)
register(custom_explanation)
print(f"Registered '{custom_explanation.name}'. Now in available('metric'): "
      f"{'debug_notebook_custom_metric' in available('metric')}")
print("\n" + custom_explanation.render(value=3.14))

Registered 'debug_notebook_custom_metric'. Now in available('metric'): True

debug_notebook_custom_metric (metric)

What it is:
  A placeholder custom metric registered from this notebook, to demonstrate the registry is extensible.

Formula:
  n/a (demo only)

How to read it:
  This exists purely to show `register()` accepts new Explanation objects at runtime.

Good vs. bad:
  n/a

Caveats:
  Not a real metric -- don't use this outside this notebook.

This result:
  demo value = 3.14


### 19.3 `explain()` -- dispatch on a string name

In [113]:
pexplain("sharpe_ratio")

sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.

Caveats:
  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.


'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.'

### 19.4 `explain()` -- dispatch on a `MetricResult` (auto-injects its own value)

In [114]:
sharpe_for_explain = portfolio.metrics.sharpe_ratio(as_result=True)
pexplain(sharpe_for_explain)

sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.

Caveats:
  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.

This result:
  good


'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.\n\nThis result:\n  good'

### 19.5 `explain()` -- dispatch on a DataFrame via `.attrs["portpy_explanation"]`

In [115]:
pexplain(comparison)  # dispatches via comparison.attrs["portpy_explanation"] == "compare_to_benchmark"

compare_to_benchmark (metric)

What it is:
  A structured comparison between a portfolio and a benchmark showing absolute performance, risk characteristics, and benchmark-relative statistics.

Formula:
  comparison = portfolio_metrics - benchmark_metrics + relative_metrics

How to read it:
  Positive differences generally indicate portfolio outperformance for return-based metrics. For risk metrics, the preferred direction depends on the objective, such as lower volatility or smaller drawdown.

Good vs. bad:
  A favorable comparison usually combines higher risk-adjusted returns, lower downside risk, positive alpha, strong information ratio, and appropriate benchmark exposure.

Caveats:
  The quality of the comparison depends on benchmark selection. A poorly chosen benchmark can make relative performance conclusions misleading.


'compare_to_benchmark (metric)\n=============================\n\nWhat it is:\n  A structured comparison between a portfolio and a benchmark showing absolute performance, risk characteristics, and benchmark-relative statistics.\n\nFormula:\n  comparison = portfolio_metrics - benchmark_metrics + relative_metrics\n\nHow to read it:\n  Positive differences generally indicate portfolio outperformance for return-based metrics. For risk metrics, the preferred direction depends on the objective, such as lower volatility or smaller drawdown.\n\nGood vs. bad:\n  A favorable comparison usually combines higher risk-adjusted returns, lower downside risk, positive alpha, strong information ratio, and appropriate benchmark exposure.\n\nCaveats:\n  The quality of the comparison depends on benchmark selection. A poorly chosen benchmark can make relative performance conclusions misleading.'

### 19.6 `explain()` -- unsupported object raises `TypeError`

In [116]:
try:
    pexplain(12345)
except TypeError as exc:
    print(f"Expected TypeError (int has no explanation dispatch): {exc}")

Expected TypeError (int has no explanation dispatch): Don't know how to explain an object of type 'int'. Pass a registered name (str), a MetricResult, or a PortPy chart/model/strategy.


### 19.7 `MetricResult` -- it's a real `float` that also knows what it means

In [117]:
print("isinstance of float:", isinstance(sharpe_for_explain, float))
print("str():  ", str(sharpe_for_explain), "  (plain number, so print() stays clean)")
print("repr(): ", repr(sharpe_for_explain), "  (name + one-line interpretation, for REPL/notebook display)")
print("arithmetic still works: sharpe_for_explain * 2 =", sharpe_for_explain * 2)
print("comparisons still work: sharpe_for_explain > 0 =", sharpe_for_explain > 0)
print(".value property:", sharpe_for_explain.value)
print(".interpretation property:", sharpe_for_explain.interpretation)
print(".unit:", sharpe_for_explain.unit, "| .name:", sharpe_for_explain.name, "| .meta:", sharpe_for_explain.meta)
print("round(sharpe_for_explain, 2) =", round(sharpe_for_explain, 2))
print("float(sharpe_for_explain) =", float(sharpe_for_explain))
sharpe_for_explain.explain()

isinstance of float: True
str():   1.0062447280036468   (plain number, so print() stays clean)
repr():  sharpe_ratio=1.00624 - good   (name + one-line interpretation, for REPL/notebook display)
arithmetic still works: sharpe_for_explain * 2 = 2.0124894560072937
comparisons still work: sharpe_for_explain > 0 = True
.value property: 1.0062447280036468
.interpretation property: good
.unit: None | .name: sharpe_ratio | .meta: {}
round(sharpe_for_explain, 2) = 1.01
float(sharpe_for_explain) = 1.0062447280036468
sharpe_ratio (metric)

What it is:
  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.

Formula:
  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)

How to read it:
  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.

Good vs. bad:
  Higher is generally better. Values above 1 are commonly con

'sharpe_ratio (metric)\n=====================\n\nWhat it is:\n  Measures excess return earned per unit of total volatility taken, using the risk-free rate as the return hurdle.\n\nFormula:\n  mean(r - rf) / std(r - rf, ddof=1) * sqrt(periods_per_year)\n\nHow to read it:\n  A Sharpe of 1.0 means the strategy generated approximately one unit of excess return for each unit of volatility. Higher values indicate better risk-adjusted performance.\n\nGood vs. bad:\n  Higher is generally better. Values above 1 are commonly considered strong, but interpretation depends on the asset class, time period, and strategy complexity.\n\nCaveats:\n  Sharpe treats all volatility as bad, including upside volatility. It can also overstate strategies with asymmetric downside risk, illiquidity, or short backtests.\n\nThis result:\n  good'

### 19.8 `portpy.explain` callable facade -- `.available()` / `.get()` / `.register()` via the top-level object

In [118]:
print("portpy.explain is callable:", callable(pexplain))
print("portpy.explain.available('metric') length:", len(pexplain.available("metric")))
print("portpy.explain.get('beta').category:", pexplain.get("beta").category)

portpy.explain is callable: True
portpy.explain.available('metric') length: 68
portpy.explain.get('beta').category: metric


## 20. `Portfolio.metrics` Auto-Fill Mechanics

`portfolio.metrics.<fn>()` is a thin, dynamically-bound wrapper around the plain function in
`portpy.metrics` -- exercised here directly so the auto-fill behavior documented on
`_MetricsNamespace` is demonstrably true, not just assumed.

### 20.1 No positional args: every eligible parameter is auto-filled

In [119]:
auto_vol = portfolio.metrics.volatility()
manual_vol = volatility(portfolio.returns(), periods_per_year=portfolio.frequency)
print("volatility(): auto-fill matches manual call:", np.isclose(auto_vol, manual_vol))

volatility(): auto-fill matches manual call: True


### 20.2 Naming one kwarg still auto-fills every other eligible parameter

In [120]:
override_vol = portfolio.metrics.volatility(periods_per_year=252)
print(f"volatility(periods_per_year=252) -> {override_vol:.2%}  (portfolio.frequency is {portfolio.frequency}, "
      f"but `returns` was still auto-filled)")

volatility(periods_per_year=252) -> 15.08%  (portfolio.frequency is 365, but `returns` was still auto-filled)


### 20.3 Parameters outside the auto-fill list (e.g. `benchmark`) must always be supplied

In [121]:
auto_beta = portfolio.metrics.beta(benchmark=benchmark_returns)
manual_beta = beta(portfolio.returns(), benchmark_returns)
print("beta(benchmark=...): auto-fill matches manual call:", np.isclose(auto_beta, manual_beta))

try:
    portfolio.metrics.beta()
except TypeError as exc:
    print(f"\nExpected TypeError (benchmark isn't auto-filled -- it's not in the auto-fill parameter list): {exc}")

beta(benchmark=...): auto-fill matches manual call: True

Expected TypeError (benchmark isn't auto-filled -- it's not in the auto-fill parameter list): beta() missing 1 required positional argument: 'benchmark'


### 20.4 Any positional argument disables auto-fill for the *entire* call

In [122]:
positional_vol = portfolio.metrics.volatility(portfolio.returns(), True)  # (returns, annualized) positionally
print(f"volatility(returns, annualized=True) positionally -> {positional_vol:.4%} "
      f"(periods_per_year silently reverts to the function's own default of 252, "
      f"not portfolio.frequency={portfolio.frequency}, because passing ANY positional arg skips auto-fill entirely)")

default_252_vol = volatility(portfolio.returns(), annualized=True, periods_per_year=252)
print(f"cross-check -- plain volatility(..., periods_per_year=252) directly = {default_252_vol:.4%} "
      f"(matches the positional call above: {np.isclose(positional_vol, default_252_vol)})")

volatility(returns, annualized=True) positionally -> 15.0827% (periods_per_year silently reverts to the function's own default of 252, not portfolio.frequency=365, because passing ANY positional arg skips auto-fill entirely)
cross-check -- plain volatility(..., periods_per_year=252) directly = 15.0827% (matches the positional call above: True)


### 20.5 `covariance_matrix` / `correlation_matrix` auto-fill from *asset-level* returns, not the portfolio aggregate

In [123]:
auto_cov = portfolio.metrics.covariance_matrix()
manual_cov = covariance_matrix(portfolio.asset_returns())
print("covariance_matrix(): auto-fill uses asset_returns(), matches manual call:",
      np.allclose(auto_cov.to_numpy(), manual_cov.to_numpy()))

covariance_matrix(): auto-fill uses asset_returns(), matches manual call: True


### 20.6 `__dir__` exposes every `portpy.metrics` name on `.metrics`

In [124]:
from portpy import metrics as metrics_module

exposed = set(dir(portfolio.metrics))
missing_from_dir = set(metrics_module.__all__) - exposed
print(f"All {len(metrics_module.__all__)} portpy.metrics.__all__ names appear in dir(portfolio.metrics): "
      f"{not missing_from_dir}")
if missing_from_dir:
    print("Missing:", missing_from_dir)

try:
    portfolio.metrics.this_metric_does_not_exist()
except AttributeError as exc:
    print(f"\nExpected AttributeError (unknown metric name): {exc}")

All 89 portpy.metrics.__all__ names appear in dir(portfolio.metrics): True

Expected AttributeError (unknown metric name): Portfolio.metrics has no 'this_metric_does_not_exist'. See portpy.metrics.__all__ for the full list.


## 21. Coverage Audit & Not-Yet-Implemented Subpackages

### 21.1 Verify every name in `portpy.metrics.__all__` was both (a) called, and (b)
explained -- not just "ran once", but "its explanation card actually renders"

In [125]:
from portpy import metrics as metrics_module

# __all__ mixes submodule names (e.g. "risk", "returns") with the actual functions.
submodule_names = {"benchmarks", "costs", "covariance", "distributions", "drawdowns",
                    "performance", "regressions", "returns", "risk", "rolling", "summary"}
all_function_names = set(metrics_module.__all__) - submodule_names

missing_covered = sorted(all_function_names - COVERED)
missing_explained = sorted(all_function_names - EXPLAINED)

print(f"portpy.metrics function count:        {len(all_function_names)}")
print(f"Called (COVERED) in this notebook:    {len(COVERED & all_function_names)}")
print(f"Explained (EXPLAINED) in this notebook: {len(EXPLAINED & all_function_names)}")

if missing_covered:
    print(f"NOT YET CALLED: {missing_covered}")
else:
    print("Every single function in portpy.metrics.__all__ was called above.")

if missing_explained:
    print(f"NOT YET EXPLAINED: {missing_explained}")
else:
    print("Every single function in portpy.metrics.__all__ was ALSO explained above "
          "(either .explain() on a live MetricResult, or portpy.explain(name) directly).")

assert not missing_covered, f"Call coverage gap: {missing_covered}"
assert not missing_explained, f"Explain coverage gap: {missing_explained}"

# Independent, brute-force cross-check: every one of these names actually has a working
# registered Explanation that renders without raising -- not just that we *called* explain().
render_failures = []
for name in sorted(all_function_names):
    try:
        get(name).render()
    except Exception as exc:  # noqa: BLE001 - we want to catch and report literally anything
        render_failures.append((name, str(exc)))
print(f"\nBrute-force get(name).render() sanity check over all {len(all_function_names)} names: "
      f"{len(render_failures)} failures")
assert not render_failures, f"Explanation cards that fail to render: {render_failures}"

portpy.metrics function count:        78
Called (COVERED) in this notebook:    78
Explained (EXPLAINED) in this notebook: 78
Every single function in portpy.metrics.__all__ was called above.
Every single function in portpy.metrics.__all__ was ALSO explained above (either .explain() on a live MetricResult, or portpy.explain(name) directly).

Brute-force get(name).render() sanity check over all 78 names: 0 failures


### 21.2 `portpy.core`, `portpy.utils.validation`, and `Portfolio` -- manual coverage checklist

(These don't share one common registry like `portpy.metrics.__all__`, so this is an explicit
checklist instead of a programmatic assertion.)

In [126]:
core_checklist = [
    "AssetClass", "ALWAYS_ON_CLASSES", "align_calendars", "calendar_coverage_report",
    "detect_frequency", "convert_to_base_currency", "equal_weights", "normalize_weights",
]
validation_checklist = [
    "ensure_datetime_index", "ensure_min_observations", "validate_confidence",
    "to_series", "align_pair", "periodic_rate_from_annual", "safe_divide",
]
portfolio_checklist = [
    "__init__", "prices", "asset_names", "num_assets", "weights", "set_weights",
    "set_risk_free_rate", "asset_returns", "returns", "price_index", "__repr__", "metrics namespace",
]
explain_checklist = [
    "Explanation", "MetricResult", "register", "get", "available", "explain (dispatch fn)",
]

for section_name, checklist in [
    ("portpy.core", core_checklist),
    ("portpy.utils.validation", validation_checklist),
    ("Portfolio", portfolio_checklist),
    ("portpy.explain", explain_checklist),
]:
    print(f"{section_name}: {len(checklist)}/{len(checklist)} exercised above -- {checklist}")

portpy.core: 8/8 exercised above -- ['AssetClass', 'ALWAYS_ON_CLASSES', 'align_calendars', 'calendar_coverage_report', 'detect_frequency', 'convert_to_base_currency', 'equal_weights', 'normalize_weights']
portpy.utils.validation: 7/7 exercised above -- ['ensure_datetime_index', 'ensure_min_observations', 'validate_confidence', 'to_series', 'align_pair', 'periodic_rate_from_annual', 'safe_divide']
Portfolio: 12/12 exercised above -- ['__init__', 'prices', 'asset_names', 'num_assets', 'weights', 'set_weights', 'set_risk_free_rate', 'asset_returns', 'returns', 'price_index', '__repr__', 'metrics namespace']
portpy.explain: 6/6 exercised above -- ['Explanation', 'MetricResult', 'register', 'get', 'available', 'explain (dispatch fn)']


### 21.3 Not-yet-implemented subpackages

`portpy.models`, `portpy.strategies`, and `portpy.visualization` are placeholder subpackages
in this PortPy version -- each `__init__.py` contains only a `# Yet to be made` comment and
exposes no public callables, so there is nothing to exercise here yet.

In [127]:
import importlib

for subpkg in ("portpy.models", "portpy.strategies", "portpy.visualization"):
    mod = importlib.import_module(subpkg)
    public_names = [n for n in dir(mod) if not n.startswith("_")]
    print(f"{subpkg}: public names = {public_names}  (empty -> not yet implemented)")

portpy.models: public names = []  (empty -> not yet implemented)
portpy.strategies: public names = []  (empty -> not yet implemented)
portpy.visualization: public names = []  (empty -> not yet implemented)


### 21.4 Final tally

In [128]:
print("=" * 70)
print("DEBUG NOTEBOOK COMPLETE")
print("=" * 70)
print(f"Portfolio tested:     {portfolio.name!r} ({portfolio.num_assets} assets, "
      f"{portfolio.frequency} periods/year)")
print(f"Asset classes used:   {sorted({c.value for c in portfolio.asset_classes.values()})}")
print(f"Benchmark:             SPY")
print(f"Risk-free rate:        {portfolio.risk_free_rate:.4%} (^IRX)")
print(f"portpy.metrics functions called:    {len(COVERED & all_function_names)}/{len(all_function_names)}")
print(f"portpy.metrics functions explained: {len(EXPLAINED & all_function_names)}/{len(all_function_names)}")
print(f"Registered explanations available:  {len(available())}")

DEBUG NOTEBOOK COMPLETE
Portfolio tested:     'All-Weather Multi-Asset Portfolio' (9 assets, 365 periods/year)
Asset classes used:   ['bond', 'commodity', 'crypto', 'equity', 'real_estate']
Benchmark:             SPY
Risk-free rate:        3.7250% (^IRX)
portpy.metrics functions called:    78/78
portpy.metrics functions explained: 78/78
Registered explanations available:  79
